In [ ]:
import os
import random
import shutil

# Paths
base_path = r"E:\Prithu\New Yolo"
train_img = os.path.join(base_path, "images", "train")
val_img = os.path.join(base_path, "images", "val")

train_lbl = os.path.join(base_path, "labels", "train")
val_lbl = os.path.join(base_path, "labels", "val")

# Create val folders if not exist
os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

# Get image list
images = [f for f in os.listdir(train_img)
          if f.lower().endswith(('.jpg', '.png', '.jpeg', '.tif', '.tiff'))]

# Select 10 random images
val_images = random.sample(images, 10)

for img in val_images:
    # Move image
    shutil.move(
        os.path.join(train_img, img),
        os.path.join(val_img, img)
    )

    # Corresponding label
    label = os.path.splitext(img)[0] + ".txt"

    lbl_src = os.path.join(train_lbl, label)
    lbl_dst = os.path.join(val_lbl, label)

    if os.path.exists(lbl_src):
        shutil.move(lbl_src, lbl_dst)
    else:
        print(f"⚠️ Label not found for {img}")

print("✅ 10 images moved from train to val")


In [ ]:
import os
import shutil
from ultralytics import YOLO

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
PROJECT_DIR = r"E:\Prithu\Yolo\dataset_v8_final"
RUN_NAME = "workstation_Medium_finetune_vnew13"  # Renamed to reflect model change
DATA_YAML = r"E:\Prithu\Yolo\dataset_v8_final\data.yaml"

# ==========================================
# 2. INITIALIZE MODEL (CHANGED)
# ==========================================
# 🔴 FIX 1: Switched from 'yolov8x-seg.pt' to 'yolov8m-seg.pt' (Medium)
# The 'X' model was overfitting your small dataset (442 imgs). 
# 'M' will generalize better for the Cathodes.
print("🧠 Loading YOLOv8-Medium Segmentation Model...")
model = YOLO('yolov8m-seg.pt') 

# ==========================================
# 3. TRAIN WITH REVISED AUGMENTATION
# ==========================================
print("🚀 Starting Workstation Training on GPU...")

results = model.train(
    data=DATA_YAML,
    
    # --- HARDWARE SETTINGS ---
    device=1,
    epochs=250,
    imgsz=1119,          # High res is good for microscopy
    
    # 🔴 FIX 2: Increased Batch Size
    # You have an RTX A6000 (48GB VRAM). Use it! 
    # Larger batches = better batch normalization = better learning stability.
    batch=32,           
    
    workers=0,
    cache=True,
    
    # --- OPTIMIZATION ---
    patience=50,
    optimizer='auto',
    cos_lr=True,
    pretrained=True,
    
    # --- ADVANCED DATA AUGMENTATION ---
    # 🔴 FIX 3: Full Rotation
    # Microscopy has no gravity. A cathode is a cathode at 180 degrees.
    # 10.0 was way too small.
    degrees=180.0,      
    
    translate=0.1,
    scale=0.6,          # Keep this, it helps with size variance
    shear=2.0,
    perspective=0.0005,
    flipud=0.5,
    fliplr=0.5,
    
    # --- COLOR JITTER (Reduced slightly) ---
    # Microscopy is sensitive to intensity. Reduced saturation/hue jitter slightly.
    hsv_h=0.01,         
    hsv_s=0.5,          
    hsv_v=0.4,
    
    # --- COMPOSITION AUGMENTATION ---
    mosaic=1.0,         # Good, keep this.
    
    # 🔴 FIX 4: Disabled Mixup
    # Mixup blends images (ghosting). For faint cells/particles, this confuses 
    # the model more than it helps on small datasets.
    mixup=0.0,          
    
    copy_paste=0.4,     # Excellent for segmentation, keep this.
    
    # --- OUTPUT ---
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True
)

# ==========================================
# 4. VALIDATION & METRICS
# ==========================================
print("\n📊 Running Final Validation to calculate IoU & mAP...")
metrics = model.val()

print("\n" + "="*40)
print("🎯 FINAL TRAINING METRICS")
print("="*40)
print(f"Mask mAP@50-95: {metrics.seg.map:.4f} (Primary Accuracy Score)")
print(f"Mask mAP@50:    {metrics.seg.map50:.4f} (Lenient Accuracy)")
print(f"Box mAP@50-95:  {metrics.box.map:.4f}")
print("="*40)

# ==========================================
# 5. SAVE WEIGHTS
# ==========================================
best_weight_path = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")
destination_path = os.path.join(os.getcwd(), "best_workstation_Medium.pt")

if os.path.exists(best_weight_path):
    shutil.copy(best_weight_path, destination_path)
    print(f"\n✅ SUCCESS! Best model saved locally as: '{destination_path}'")
    print(f"Use this path in your analysis code: r'{destination_path}'")
else:
    print(f"\n⚠️ Could not move file. Check: {best_weight_path}")

In [ ]:
import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ==========================================
# 1. CONFIGURATION
# ==========================================
# 🔴 UPDATE THESE PATHS IF NEEDED
IMAGE_PATH = r"E:\Prithu\new images\251203_HS25020-04_R2_Kachel_20x.tif"
MODEL_PATH = r'c:\Users\06877\AppData\Local\Programs\Microsoft VS Code\best_workstation_Medium.pt'
OUTPUT_PATH = r"E:\Prithu\new output\ANALYZED_METROLOGY_SMART_MERGE251203_HS25020-04_R2_Kachel_20x.jpg"

# TILING SETTINGS
TILE_SIZE = 960       # AI sees this size
OVERLAP = 200         # Ensures no objects are cut in half at edges
CONF_THRESHOLD = 0.15 # Lower threshold to catch faint objects

# CALIBRATION
MICRONS_PER_PIXEL = 0.22 
UNIT_LABEL = "um"
FONT_SCALE_M = 1.5
THICKNESS = 4
DOT_RADIUS = 15
MAX_OBJECTS = 100
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def get_accurate_top_and_tilt(contour):
    """
    Calculates the accurate top point and tilt angle of a contour.
    Includes fixes for Numpy DeprecationWarnings.
    """
    # Flatten just in case
    pts = contour.reshape(-1, 2)
    
    if len(pts) < 5: 
        return 0, pts[0], pts[0]
    
    # Fit line
    [vx, vy, vx_c, vy_c] = cv2.fitLine(pts, cv2.DIST_L2, 0, 0.01, 0.01)
    
    # 🔴 FIX: Convert numpy arrays to float python scalars to avoid DeprecationWarning
    vx_c = float(vx_c)
    vy_c = float(vy_c)
    vx = float(vx)
    vy = float(vy)

    # Find top and bottom Y points
    top_idx = np.argmin(pts[:, 1])
    bot_idx = np.argmax(pts[:, 1])
    top_y = float(pts[top_idx, 1])
    bot_y = float(pts[bot_idx, 1])
    
    # Project X based on the fitted line slope
    if abs(vy) > 1e-5:
        slope = vx / vy
        top_x_proj = int(vx_c + slope * (top_y - vy_c))
        bot_x_projected = int(vx_c + slope * (bot_y - vy_c))
        accurate_top = np.array([top_x_proj, int(top_y)])
        accurate_bot = np.array([bot_x_projected, int(bot_y)])
    else:
        # Vertical line case
        accurate_top = np.array([int(vx_c), int(top_y)])
        accurate_bot = np.array([int(vx_c), int(bot_y)])

    # Calculate Angle
    dx = accurate_top[0] - accurate_bot[0]
    dy = accurate_top[1] - accurate_bot[1]
    angle_rad = math.atan2(dy, dx)
    angle_deg = math.degrees(angle_rad)
    deviation = 90 - abs(angle_deg + 90)
    
    return abs(deviation), accurate_bot, accurate_top

def combine_lane_fragments(group):
    """
    Takes a group of contours (broken pieces, overlaps) that belong to one electrode
    and wraps them in a single Convex Hull.
    """
    # Stack all points from all overlapping rectangles/broken lines
    all_pts = np.vstack([g['contour'].reshape(-1, 2) for g in group])
    
    # Create ONE smooth outer shell (Convex Hull) around all pieces
    hull = cv2.convexHull(all_pts)
    
    # Recalc metrics on the clean hull
    _, _, top_point = get_accurate_top_and_tilt(hull)
    
    # Return the clean merged object
    return {
        'label': group[0]['label'],
        'contour': hull,
        'top': top_point,
        'mean_x': np.mean(hull[:, 0, 0])
    }

def smart_merge_electrodes(raw_structures, lane_width=60):
    """
    Groups all detections (fragments, duplicates) that belong to the same vertical 'lane'.
    Then creates a single Convex Hull wrapping them all.
    
    lane_width: Max distance (pixels) to consider parts as the same electrode.
    """
    if not raw_structures: return []

    # 1. Calculate centroid X for every fragment
    for s in raw_structures:
        pts = s['contour'].reshape(-1, 2)
        s['center_x'] = np.mean(pts[:, 0])

    # 2. Sort all fragments left-to-right
    raw_structures.sort(key=lambda s: s['center_x'])

    merged_output = []
    current_lane = [raw_structures[0]]

    # 3. Group items that are close horizontally
    for i in range(1, len(raw_structures)):
        curr = raw_structures[i]
        
        # Calculate the average X of the current lane we are building
        lane_mean_x = np.mean([item['center_x'] for item in current_lane])
        
        # If the new part is within the lane width AND implies same class
        # (Using mean_x distance is more robust than bounding box overlap for thin objects)
        if abs(curr['center_x'] - lane_mean_x) < lane_width and curr['label'] == current_lane[0]['label']:
            current_lane.append(curr)
        else:
            # Lane finished -> Combine everything in it
            merged_output.append(combine_lane_fragments(current_lane))
            current_lane = [curr] # Start new lane
    
    # Append the final lane
    if current_lane:
        merged_output.append(combine_lane_fragments(current_lane))

    return merged_output[:MAX_OBJECTS]

# ==========================================
# 3. MAIN ANALYSIS
# ==========================================
def run_metrology():
    print("🚀 STARTING HIGH-RES ANALYSIS...")
    
    # 1. Load FULL RESOLUTION Image
    print("🖼️ Loading Original Image...")
    img = cv2.imread(IMAGE_PATH) # Loads 8192x... directly
    if img is None:
        print("❌ Error: Could not load image.")
        return
        
    h_img, w_img, _ = img.shape
    print(f"   Original Dimensions: {w_img} x {h_img} pixels (Full Res)")
    
    model = YOLO(MODEL_PATH)
    
    # 2. Process in Tiles (But keep coordinates Global)
    stride = TILE_SIZE - OVERLAP
    all_raw_structures = []
    total_tiles = math.ceil(h_img/stride) * math.ceil(w_img/stride)
    count = 0
    
    print(f"⚡ Tiling & Predicting ({total_tiles} chunks)...")

    for y in range(0, h_img, stride):
        for x in range(0, w_img, stride):
            count += 1
            # Define crop coordinates
            x_start = min(x, max(0, w_img - TILE_SIZE))
            y_start = min(y, max(0, h_img - TILE_SIZE))
            
            # Crop the tile (No resizing here!)
            tile = img[y_start:y_start+TILE_SIZE, x_start:x_start+TILE_SIZE]
            
            # Predict on the tile
            results = model(tile, conf=CONF_THRESHOLD, imgsz=TILE_SIZE, verbose=False, device=0)
            
            if results[0].masks:
                for i, polygon in enumerate(results[0].masks.xy):
                    if len(polygon) < 1: continue
                    
                    cls_id = int(results[0].boxes.cls[i].item())
                    label = CLASS_MAP.get(cls_id, 'Unknown')
                    
                    # --- CRITICAL STEP: Shift back to Global Coordinates ---
                    pts = np.array(polygon, np.int32)
                    pts[:, 0] += x_start  # Add tile offset X
                    pts[:, 1] += y_start  # Add tile offset Y
                    
                    if len(pts) < 5: continue
                    _, _, top_point = get_accurate_top_and_tilt(pts)
                    
                    all_raw_structures.append({
                        'label': label, 
                        'contour': pts, # This contour is now in GLOBAL coordinates
                        'top': top_point, 
                        'x': top_point[0]
                    })
            if count % 10 == 0: print(f"   Processed {count}/{total_tiles} tiles...", end='\r')

    print(f"\n✅ Detections Found: {len(all_raw_structures)}")
    
    # 3. Merge (NEW SMART LOGIC)
    print("⚙️ Merging Broken Parts & Overlaps...")
    # lane_width=60 works well for 200x zoom to snap overlaps together
    structures = smart_merge_electrodes(all_raw_structures, lane_width=60) 
    print(f"✅ Final Electrodes: {len(structures)}")

    # 4. Draw on FULL RES Image
    print("🎨 Drawing on Full Resolution Image...")
    colors = {'Anode': (0, 255, 0), 'Cathode': (0, 140, 255)} 
    
    # Sort Left-to-Right for measurement
    structures.sort(key=lambda s: s['mean_x'])

    for i, s in enumerate(structures):
        # Draw Contour
        color = colors.get(s['label'], (255, 255, 255))
        cv2.polylines(img, [s['contour']], isClosed=True, color=color, thickness=THICKNESS)
        cv2.circle(img, s['top'], DOT_RADIUS, (0, 0, 255), -1)
        
        # Tilt Angle
        tilt_deg, p_bot, p_top = get_accurate_top_and_tilt(s['contour'])
        cv2.line(img, p_bot, p_top, (0, 255, 255), 2)
        cv2.putText(img, f"{tilt_deg:.1f}", (p_top[0]-25, p_top[1]-60), 
                    cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_M, (0, 255, 255), THICKNESS)

        # Measure Distance: Cathode -> Anode
        if i < len(structures) - 1:
            next_s = structures[i+1]
            if s['label'] == 'Cathode' and next_s['label'] == 'Anode':
                p_cat = s['top']
                p_anode = next_s['top']
                
                cv2.line(img, p_cat, p_anode, (255, 0, 255), THICKNESS)
                dist_px = math.sqrt((p_cat[0]-p_anode[0])**2 + (p_cat[1]-p_anode[1])**2)
                real_dist = dist_px * MICRONS_PER_PIXEL
                
                mid_x = (p_cat[0] + p_anode[0]) // 2
                mid_y = (p_cat[1] + p_anode[1]) // 2
                cv2.putText(img, f"{real_dist:.1f} {UNIT_LABEL}", (mid_x, mid_y), 
                            cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_M, (255, 255, 255), THICKNESS)

    # 5. Save Full Resolution
    cv2.imwrite(OUTPUT_PATH, img)
    print(f"💾 Saved FULL SIZE Image to: {OUTPUT_PATH}")
    print("   (Go to your folder and open this file to zoom in)")

    # 6. Show Preview (Small version only for screen)
    plt.figure(figsize=(12, 8))
    display_img = cv2.resize(img, (0,0), fx=0.1, fy=0.1) 
    plt.imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Preview (Original File is saved to disk)", fontsize=18)
    plt.show()

if __name__ == "__main__":
    run_metrology()

In [ ]:
import os
import cv2
import numpy as np
import random
import shutil
from tqdm import tqdm
from glob import glob

# ==========================================
# 1. CONFIGURATION
# ==========================================
# 🔴 Path to your original 183 LARGE images & Sreeni labels
SOURCE_IMAGES = r"E:\Prithu\Final Yolo\images\train" 
SOURCE_LABELS = r"E:\Prithu\Final Yolo\labels\train"

# 🟢 Output location (Will be created)
OUTPUT_BASE = r"E:\Prithu\Final Yolo_TILED"

# Settings
TILE_SIZE = 640
OVERLAP = 128
STRIDE = TILE_SIZE - OVERLAP
VAL_SPLIT = 0.10
SEED = 42

# ==========================================
# 2. SETUP
# ==========================================
random.seed(SEED)

def setup_directories():
    if os.path.exists(OUTPUT_BASE):
        print(f"♻️ Cleaning old folder: {OUTPUT_BASE}")
        try:
            shutil.rmtree(OUTPUT_BASE)
        except PermissionError:
            print("❌ Error: Close any open images/files in the folder and try again.")
            exit()
            
    for split in ['train', 'val']:
        os.makedirs(os.path.join(OUTPUT_BASE, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_BASE, 'labels', split), exist_ok=True)

# ==========================================
# 3. POLYGON MATH
# ==========================================
def process_polygon(poly_line, crop_x, crop_y, tile_w, tile_h, img_w, img_h):
    parts = list(map(float, poly_line.strip().split()))
    class_id = int(parts[0])
    coords = parts[1:]
    
    # Reshape to (N, 2)
    points = np.array(coords).reshape(-1, 2)
    
    # 1. Denormalize (Global Pixels)
    points[:, 0] *= img_w
    points[:, 1] *= img_h
    
    # 2. Check if polygon is inside this tile
    p_min_x, p_min_y = np.min(points, axis=0)
    p_max_x, p_max_y = np.max(points, axis=0)
    
    if (p_max_x < crop_x) or (p_min_x > crop_x + tile_w) or \
       (p_max_y < crop_y) or (p_min_y > crop_y + tile_h):
        return None

    # 3. Shift to Tile Coordinates
    points[:, 0] -= crop_x
    points[:, 1] -= crop_y
    
    # 4. Clip to Tile Edges
    points[:, 0] = np.clip(points[:, 0], 0, tile_w)
    points[:, 1] = np.clip(points[:, 1], 0, tile_h)
    
    # 5. Normalize (0-1) for YOLO
    points[:, 0] /= tile_w
    points[:, 1] /= tile_h
    
    # Format back to string
    flat_points = points.flatten().tolist()
    str_points = " ".join([f"{p:.6f}" for p in flat_points])
    return f"{class_id} {str_points}"

# ==========================================
# 4. SLICING LOGIC
# ==========================================
def slice_dataset():
    # Gather ALL images (including TIFs)
    image_files = glob(os.path.join(SOURCE_IMAGES, "*.*"))
    valid_exts = ['.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp']
    image_files = [f for f in image_files if os.path.splitext(f)[1].lower() in valid_exts]
    
    if not image_files:
        print(f"❌ No images found in {SOURCE_IMAGES}")
        return

    # Shuffle for Train/Val split
    random.shuffle(image_files)
    split_idx = int(len(image_files) * VAL_SPLIT)
    val_files = image_files[:split_idx]
    train_files = image_files[split_idx:]
    
    print(f"📊 Processing: {len(train_files)} Train | {len(val_files)} Val")

    for split, files in [('train', train_files), ('val', val_files)]:
        print(f"🔪 Slicing {split} set...")
        
        for img_path in tqdm(files):
            # Force load as grayscale
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            
            # 🟢 FIXED LINE: Handle both (H,W) and (H,W,C) shapes
            h_img, w_img = img.shape[:2]
            
            stem = os.path.splitext(os.path.basename(img_path))[0]
            label_path = os.path.join(SOURCE_LABELS, stem + ".txt")
            
            # Read Polygons
            raw_lines = []
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    raw_lines = f.readlines()

            # Slice Loop
            tile_count = 0
            for y in range(0, h_img, STRIDE):
                for x in range(0, w_img, STRIDE):
                    x_end = min(x + TILE_SIZE, w_img)
                    y_end = min(y + TILE_SIZE, h_img)
                    x_start = max(0, x_end - TILE_SIZE)
                    y_start = max(0, y_end - TILE_SIZE)
                    
                    # Crop Image
                    tile = img[y_start:y_end, x_start:x_end]
                    
                    # Process Labels
                    new_labels = []
                    for line in raw_lines:
                        processed = process_polygon(line, x_start, y_start, TILE_SIZE, TILE_SIZE, w_img, h_img)
                        if processed:
                            new_labels.append(processed)
                    
                    # Save Tile
                    save_name = f"{stem}_t{tile_count}"
                    out_img_path = os.path.join(OUTPUT_BASE, 'images', split, f"{save_name}.png")
                    cv2.imwrite(out_img_path, tile)
                    
                    # Save Label (Only if polygons exist in this tile)
                    if new_labels:
                        out_lbl_path = os.path.join(OUTPUT_BASE, 'labels', split, f"{save_name}.txt")
                        with open(out_lbl_path, 'w') as f:
                            f.write('\n'.join(new_labels))
                            
                    tile_count += 1

    # Write data.yaml
    yaml_content = f"""path: {OUTPUT_BASE}
train: images/train
val: images/val
nc: 2
names:
  0: Cathode
  1: Anode
"""
    with open(os.path.join(OUTPUT_BASE, "data.yaml"), "w") as f:
        f.write(yaml_content)
    
    print("\n✅ DATASET READY WITH POLYGONS!")

if __name__ == "__main__":
    setup_directories()
    slice_dataset()

In [ ]:
import os
import shutil
from ultralytics import YOLO

# ==========================================
# 1. CONFIGURATION
# ==========================================
# 🟢 Ensure this points to your Tiled 640px dataset
DATA_YAML = r"E:\Prithu\Final Yolo_TILED\data.yaml"

PROJECT_DIR = r"E:\Prithu\Yolo\Final_Training_Run"
RUN_NAME = "Microscopy_Vertical_Detect_Final"

# ==========================================
# 2. MAIN EXECUTION
# ==========================================
if __name__ == '__main__':
    # --------------------------------------
    # STEP 1: LOAD MODEL
    # --------------------------------------
    print("🧠 Loading YOLOv8-Medium DETECTION model...")
    # Using Detection (.pt) because your labels are boxes
    model = YOLO("yolov8m.pt") 

    # --------------------------------------
    # STEP 2: TRAIN (Vertical Optimized)
    # --------------------------------------
    print(f"🚀 Starting Vertical-Only Training...")
    model.train(
        data=DATA_YAML,
        
        # --- HARDWARE ---
        device=1,           # A6000 (GPU 1)
        epochs=250,
        imgsz=640,          # Matched to tiles
        batch=64,           
        workers=8,
        cache=True,
        
        # --- OPTIMIZATION ---
        optimizer="AdamW",
        lr0=0.001,
        patience=50,
        cos_lr=True,
        
        # --- VERTICAL SPECIFIC AUGMENTATIONS ---
        # 🛑 STRICTLY VERTICAL SETTINGS
        degrees=0.0,        # ❌ No rotation (Forces vertical learning)
        shear=0.0,          # ❌ No shearing (Prevents slanting)
        perspective=0.0,    # ❌ No perspective warp
        
        # ✅ ALLOWED AUGMENTATIONS
        fliplr=0.5,         # OK: Mirroring L/R keeps it vertical
        flipud=0.5,         # OK: Mirroring U/D keeps it vertical
        scale=0.5,          # OK: Zooming +/- 50% is safe
        
        # ❌ GRAYSCALE SAFETY
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        
        # ✅ STRUCTURE (Prevents "broken" electrodes)
        mosaic=1.0,         
        mixup=0.1,
        copy_paste=0.0,     # Disabled for detection
        
        # --- OUTPUT ---
        project=PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True
    )

    # --------------------------------------
    # STEP 3: VALIDATION
    # --------------------------------------
    print("\n📊 Running Validation...")
    
    metrics = model.val(
        data=DATA_YAML,
        split='val',        
        imgsz=640,
        batch=64,
        device=1,
        
        save=True,          
        plots=True,         
        save_conf=True,     
        conf=0.001,         
        iou=0.6,            
        max_det=300,        
        
        project=PROJECT_DIR,
        name=f"{RUN_NAME}_VALIDATION"
    )

    # --------------------------------------
    # STEP 4: REPORT & SAVE
    # --------------------------------------
    print("\n🎯 VALIDATION METRICS")
    print(f"Box mAP@50-95: {metrics.box.map:.4f}")
    print(f"Box mAP@50:    {metrics.box.map50:.4f}")
    
    # Save to current folder
    best_weight_source = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")
    local_save_path = os.path.join(os.getcwd(), "best_vertical_detect.pt")
    
    if os.path.exists(best_weight_source):
        shutil.copy(best_weight_source, local_save_path)
        print(f"\n✅ Best model saved locally as: {local_save_path}")
    else:
        print("\n❌ Could not find 'best.pt'.")

In [ ]:
# --- FORCE IMAGE DISPLAY ---
%matplotlib inline 

import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\251202_HS25020-02_R2_Kachel_20x.tif"
MODEL_PATH = r"c:\Users\06877\AppData\Local\Programs\Microsoft VS Code\best_vertical_detect.pt"
OUTPUT_PATH = r"E:\Prithu\final new output\251202_HS25020-02_R2_Kachel_20x_output.png"

# ✅ CORRECT SCALE 
MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}

# ==========================================
# 2. OVERLAP STITCHING & HULL RECONSTRUCTION
# ==========================================
def horizontal_overlap(box1, box2, buffer=10):
    """
    Returns True if the two boxes overlap horizontally.
    'buffer' expands the box slightly to catch barely-touching pieces.
    """
    # box = [x1, y1, x2, y2]
    # Check if Range1 (x1, x2) overlaps with Range2 (x1, x2)
    left1, right1 = box1[0] - buffer, box1[2] + buffer
    left2, right2 = box2[0] - buffer, box2[2] + buffer
    
    return max(left1, left2) < min(right1, right2)

def create_hull_from_stack(stack, label):
    """
    Creates a Convex Hull Polygon from all boxes in the stack.
    This 'heals' breaks by wrapping everything in one shape.
    """
    all_points = []
    
    # Collect all 4 corners from every box in the stack
    for item in stack:
        x1, y1, x2, y2 = item['box']
        all_points.append([x1, y1])
        all_points.append([x2, y1])
        all_points.append([x2, y2])
        all_points.append([x1, y2])
    
    # Generate Convex Hull
    points_array = np.array(all_points, dtype=np.int32)
    hull = cv2.convexHull(points_array)

    # Find Top Center for measuring
    # We use the highest point in the hull
    top_y_idx = np.argmin(hull[:, 0, 1])
    top_point = tuple(hull[top_y_idx][0])
    
    # Calculate bounding box of the hull for labeling
    x, y, w, h = cv2.boundingRect(hull)
    
    return {
        'label': label,
        'contour': hull,
        'box': np.array([x, y, x+w, y+h]), # Approximate box
        'top': top_point,
        'x': top_point[0]
    }

def smart_overlap_merge(raw_detections):
    """
    Merges fragments based on strict horizontal overlap.
    """
    final_objects = []

    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        # 1. Sort Top to Bottom
        fragments.sort(key=lambda d: d['box'][1])

        active_chains = [] 

        for frag in fragments:
            best_chain = None
            
            # Find a chain this fragment overlaps with
            for chain in active_chains:
                # Compare with the LAST piece of the chain
                last_piece = chain[-1]
                
                # Overlap Check
                if horizontal_overlap(last_piece['box'], frag['box'], buffer=20):
                    best_chain = chain
                    break
            
            if best_chain:
                best_chain.append(frag)
            else:
                active_chains.append([frag])

        # Fuse chains into Hulls
        for chain in active_chains:
            final_objects.append(create_hull_from_stack(chain, cls_name))

    # Final Sort Left-to-Right
    final_objects.sort(key=lambda d: d['x'])
    return final_objects

# ==========================================
# 3. SLIDING WINDOW INFERENCE
# ==========================================
def run_sliding_window_inference(model, color_img):
    gray_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY)
    h_img, w_img = gray_img.shape
    
    TILE_SIZE = 640  
    OVERLAP = 128    
    STRIDE = TILE_SIZE - OVERLAP
    all_detections = []
    
    x_steps = list(range(0, w_img, STRIDE))
    y_steps = list(range(0, h_img, STRIDE))
    
    print(f"⚡ Scanning {w_img}x{h_img}...")

    for y in tqdm(y_steps):
        for x in x_steps:
            x_end = min(w_img, x + TILE_SIZE)
            y_end = min(h_img, y + TILE_SIZE)
            x_start = x 
            y_start = y
            
            if x_end - x_start < 100 or y_end - y_start < 100: continue

            tile_1ch = gray_img[y_start:y_end, x_start:x_end]
            tile_3ch = cv2.cvtColor(tile_1ch, cv2.COLOR_GRAY2BGR)
            
            # Low confidence to get ALL pieces
            results = model(tile_3ch, conf=0.15, imgsz=640, verbose=False, device=0)
            
            if not results[0].boxes: continue

            for box in results[0].boxes:
                coords = box.xyxy[0].cpu().numpy()
                cls_id = int(box.cls[0].item())
                label = CLASS_MAP.get(cls_id, 'Unknown')
                
                x1, y1, x2, y2 = coords
                global_box = np.array([x1 + x_start, y1 + y_start, x2 + x_start, y2 + y_start])
                
                all_detections.append({
                    'label': label,
                    'box': global_box
                })
                
    return all_detections

# ==========================================
# 4. MAIN PIPELINE
# ==========================================
def run_metrology():
    print("🚀 CELL STARTED...")
    
    if not os.path.exists(MODEL_PATH): print("❌ Model not found"); return
    img_color = cv2.imread(IMAGE_PATH) 
    if img_color is None: print("❌ Image not found"); return
    
    model = YOLO(MODEL_PATH)

    # 1. Inference
    raw_fragments = run_sliding_window_inference(model, img_color)
    if not raw_fragments: print("⚠️ No objects detected."); return

    # 2. MERGE
    print("🧵 Stitching with Convex Hulls...")
    structures = smart_overlap_merge(raw_fragments)
    print(f"✅ Final Count: {len(structures)} Electrodes")

    # 3. DRAWING
    colors = {'Anode': (0, 255, 0), 'Cathode': (0, 140, 255)} 
    all_tops_y = []

    for i, s in enumerate(structures):
        all_tops_y.append(s['top'][1])
        color = colors.get(s['label'], (255, 255, 255))
        
        # Draw Convex Hull (The smooth wrapper)
        cv2.polylines(img_color, [s['contour']], isClosed=True, color=color, thickness=3)
        cv2.circle(img_color, s['top'], DOT_RADIUS, (0, 0, 255), -1)

        # Distance Logic (Cathode -> Anode)
        if i < len(structures) - 1:
            next_s = structures[i+1]
            
            if s['label'] == 'Cathode' and next_s['label'] == 'Anode':
                p1 = s['top']
                p2 = next_s['top']
                
                cv2.line(img_color, p1, p2, (255, 0, 255), 3)
                
                dist_px = math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)
                real_dist = dist_px * MICRONS_PER_PIXEL
                
                mid_x = (p1[0] + p2[0]) // 2
                mid_y = (p1[1] + p2[1]) // 2
                
                text = f"{real_dist:.1f}"
                (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.1, 2)
                cv2.rectangle(img_color, (mid_x, mid_y - th - 5), (mid_x + tw, mid_y + 5), (0,0,0), -1)
                cv2.putText(img_color, text, (mid_x, mid_y), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2)

    # Reference Line
    if all_tops_y:
        y_max = max(all_tops_y)
        cv2.line(img_color, (0, y_max), (img_color.shape[1], y_max), (0, 255, 0), 2)

    cv2.imwrite(OUTPUT_PATH, img_color)
    print(f"💾 Saved Result to: {OUTPUT_PATH}")
    
    plt.figure(figsize=(18, 14))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Final Hull Stitching", fontsize=18)
    plt.show()

# --- RUN ---
run_metrology()

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\a.tif"
import cv2
import numpy as np
import math
import os
import torch
import matplotlib.pyplot as plt
from torchvision.ops import nms
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\250115_HS24020-01_R1_Kachel_200x_Messungen-01.tif"
MODEL_PATH = r"E:\Prithu\final new output\runs\segment\train\weights\best.pt" 
OUTPUT_PATH = r"E:\Prithu\final new output\a1.png"

# ✅ SCALING & VISUALS
MICRONS_PER_PIXEL = 1.72633 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 5

# ==========================================
# 2. CORE LOGIC FUNCTIONS
# ==========================================

def apply_global_nms(detections, iou_threshold=0.3):
    """Filters redundant boxes caused by overlapping sliding windows."""
    if not detections:
        return []
    
    # Prepare tensors for NMS
    boxes = torch.tensor([d['box'] for d in detections], dtype=torch.float32)
    scores = torch.tensor([d['score'] for d in detections], dtype=torch.float32)
    
    # Perform NMS (Non-Maximum Suppression)
    keep_indices = nms(boxes, scores, iou_threshold)
    return [detections[i] for i in keep_indices]

def get_precise_tip(hull, box):
    """Identifies the top-most average point of an electrode fragment."""
    x1, y1, x2, y2 = box
    h = y2 - y1
    points = hull[:, 0, :]
    
    # Isolate top 15% for tip detection
    roi_height = min(60, h * 0.15) 
    cutoff_y = y1 + roi_height
    top_zone_points = points[points[:, 1] <= cutoff_y]
    
    if len(top_zone_points) == 0:
        top_zone_points = points

    min_y = np.min(top_zone_points[:, 1])
    peak_region = top_zone_points[top_zone_points[:, 1] <= (min_y + 3)]
    
    avg_x = int(np.mean(peak_region[:, 0]))
    avg_y = int(np.mean(peak_region[:, 1]))
    return (avg_x, avg_y)

def create_hull_from_stack(stack, label):
    """Merges multiple fragments into a single geometric structure."""
    all_points = []
    for item in stack:
        x1, y1, x2, y2 = item['box']
        all_points.extend([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
    
    points_array = np.array(all_points, dtype=np.int32)
    hull = cv2.convexHull(points_array)
    
    x, y, w, h = cv2.boundingRect(hull)
    full_box = [x, y, x+w, y+h]
    top_point = get_precise_tip(hull, full_box)
        
    return {
        'label': label,
        'contour': hull,
        'box': full_box,
        'top': top_point,
        'x': top_point[0]
    }

def smart_vertical_stitch(raw_detections):
    """Groups fragments into vertical columns (electrodes)."""
    final_objects = []

    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        fragments.sort(key=lambda d: d['box'][1]) 
        active_chains = [] 

        for frag in fragments:
            best_chain_idx = -1
            best_score = -float('inf') 
            
            for i, chain in enumerate(active_chains):
                last_piece = chain[-1]
                gap = frag['box'][1] - last_piece['box'][3]
                
                max_allowed_gap = 400 if cls_name == 'Cathode' else 250
                if gap > max_allowed_gap: continue 
                
                # Check X-overlap and alignment
                t_center = (frag['box'][0] + frag['box'][2]) / 2
                b_center = (last_piece['box'][0] + last_piece['box'][2]) / 2
                center_dist = abs(t_center - b_center)
                
                if center_dist < 50: # Alignment threshold
                    score = -center_dist 
                    if score > best_score:
                        best_score = score
                        best_chain_idx = i

            if best_chain_idx != -1:
                active_chains[best_chain_idx].append(frag)
            else:
                active_chains.append([frag])

        for chain in active_chains:
            total_h = chain[-1]['box'][3] - chain[0]['box'][1]
            if len(chain) == 1 and total_h < 30: continue 
            final_objects.append(create_hull_from_stack(chain, cls_name))

    final_objects.sort(key=lambda d: d['x']) 
    return final_objects

# ==========================================
# 3. PIPELINE EXECUTION
# ==========================================

def run_metrology():
    print("🚀 Initializing Metrology Pipeline...")
    
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Model not found at {MODEL_PATH}"); return
    
    img_color = cv2.imread(IMAGE_PATH)
    if img_color is None:
        print(f"❌ Image not found at {IMAGE_PATH}"); return
    
    model = YOLO(MODEL_PATH)
    h_img, w_img = img_color.shape[:2]
    
    # 1. SLIDING WINDOW INFERENCE
    TILE_SIZE, OVERLAP = 640, 160 
    STRIDE = TILE_SIZE - OVERLAP
    raw_fragments = []

    print(f"⚡ Scanning image ({w_img}x{h_img})...")
    for y in tqdm(range(0, h_img, STRIDE)):
        for x in range(0, w_img, STRIDE):
            x_end, y_end = min(w_img, x + TILE_SIZE), min(h_img, y + TILE_SIZE)
            tile = img_color[y:y_end, x:x_end]
            
            results = model(tile, conf=0.25, imgsz=640, verbose=False)
            
            for r in results:
                for box in r.boxes:
                    coords = box.xyxy[0].cpu().numpy()
                    raw_fragments.append({
                        'label': CLASS_MAP.get(int(box.cls[0]), 'Unknown'),
                        'box': [coords[0]+x, coords[1]+y, coords[2]+x, coords[3]+y],
                        'score': box.conf[0].item()
                    })

    # 2. CLEANUP & STITCHING
    print("🧹 Cleaning up overlaps and stitching fragments...")
    filtered_fragments = apply_global_nms(raw_fragments)
    structures = smart_vertical_stitch(filtered_fragments)

    # 3. MEASUREMENT & VISUALIZATION
    print("📐 Calculating Pythagorean distances...")
    font = cv2.FONT_HERSHEY_SIMPLEX
    
    for i in range(len(structures) - 1):
        cur, nxt = structures[i], structures[i+1]
        
        # Draw outlines
        color = (0, 255, 0) if cur['label'] == 'Anode' else (0, 140, 255)
        cv2.polylines(img_color, [cur['contour']], True, color, 2)
        cv2.circle(img_color, cur['top'], DOT_RADIUS, (0, 0, 255), -1)

        # Logic: Measure distance from Cathode to next Anode
        if cur['label'] == 'Cathode' and nxt['label'] == 'Anode':
            x1, y1 = cur['top']
            x2, y2 = nxt['top']
            
            # Pythagorean Theorem
            pixel_dist = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            micron_dist = pixel_dist * MICRONS_PER_PIXEL
            
            # Draw visual line and label
            cv2.line(img_color, (x1, y1), (x2, y2), (255, 0, 255), 2)
            
            label_text = f"{micron_dist:.1f} um"
            t_x, t_y = (x1 + x2) // 2, (y1 + y2) // 2 - 20
            cv2.putText(img_color, label_text, (t_x, t_y), font, 0.7, (255, 255, 255), 2)

    # Save and Show
    cv2.imwrite(OUTPUT_PATH, img_color)
    plt.figure(figsize=(15, 10))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.title("Metrology Result: Clean Detections & Pythagorean Distance")
    plt.axis('off')
    plt.show()

if __name__ == "__main__":
    run_metrology()
OUTPUT_PATH = r"E:\Prithu\final new output\a1.png"

# ✅ SCALING & VISUALS
MICRONS_PER_PIXEL = 1.72633 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 4

# ==========================================
# 2. GEOMETRY & STRICT STITCHING
# ==========================================
def get_x_overlap_and_center_dist(box_top, box_bottom):
    tx1, ty1, tx2, ty2 = box_top
    bx1, by1, bx2, by2 = box_bottom
    
    overlap_start = max(tx1, bx1)
    overlap_end = min(tx2, bx2)
    overlap_len = max(0, overlap_end - overlap_start)
    
    top_center = (tx1 + tx2) / 2
    btm_center = (bx1 + bx2) / 2
    center_dist = abs(top_center - btm_center)
    
    return overlap_len, center_dist

def get_precise_tip(hull, box):
    x, y, w, h = box
    points = hull[:, 0, :]
    
    # Isolate top 15% for tip detection
    roi_height = min(60, h * 0.15) 
    cutoff_y = y + roi_height
    top_zone_points = points[points[:, 1] <= cutoff_y]
    
    if len(top_zone_points) == 0:
        top_zone_points = points

    min_y = np.min(top_zone_points[:, 1])
    peak_region = top_zone_points[top_zone_points[:, 1] <= (min_y + 3)]
    
    if len(peak_region) > 0:
        avg_x = int(np.mean(peak_region[:, 0]))
        avg_y = int(np.mean(peak_region[:, 1]))
        return (avg_x, avg_y)
    
    return tuple(points[np.argmin(points[:, 1])])

def create_hull_from_stack(stack, label):
    all_points = []
    for item in stack:
        x1, y1, x2, y2 = item['box']
        all_points.extend([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
    
    points_array = np.array(all_points, dtype=np.int32)
    hull = cv2.convexHull(points_array)
    
    x, y, w, h = cv2.boundingRect(hull)
    full_box = np.array([x, y, x+w, y+h])
    top_point = get_precise_tip(hull, full_box)
        
    return {
        'label': label,
        'contour': hull,
        'box': full_box,
        'top': top_point,
        'x': top_point[0]
    }

def smart_vertical_stitch(raw_detections):
    """STRICT VERTICAL LOCK: Merges pieces into a single electrode body before tip detection."""
    final_objects = []

    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        # Sort Top to Bottom to find vertical chains
        fragments.sort(key=lambda d: d['box'][1]) 
        active_chains = [] 

        for frag in fragments:
            best_chain_idx = -1
            best_score = -float('inf') 
            frag_y_top = frag['box'][1]
            
            for i, chain in enumerate(active_chains):
                last_piece = chain[-1]
                gap = frag_y_top - last_piece['box'][3]
                
                # RELAXED GAP: Increased to 400 for Cathodes to bridge breakages
                # strictly along the vertical axis.
                max_allowed_gap = 400 if cls_name == 'Cathode' else 250
                if gap > max_allowed_gap: continue 
                
                overlap_len, center_dist = get_x_overlap_and_center_dist(frag['box'], last_piece['box'])
                
                if overlap_len > 0:
                    # Still use a high center-alignment penalty to avoid jumping columns
                    score = overlap_len - (center_dist * 4.0) 
                    if score > best_score:
                        best_score = score
                        best_chain_idx = i

            if best_chain_idx != -1:
                active_chains[best_chain_idx].append(frag)
            else:
                active_chains.append([frag])

        # After grouping all broken pieces, create one single Hull for the whole electrode
        for chain in active_chains:
            # Filter minor noise
            total_h = chain[-1]['box'][3] - chain[0]['box'][1]
            if len(chain) == 1 and total_h < 30:
                continue 
            
            # create_hull_from_stack will now find the 'True Top' of the combined pieces
            final_objects.append(create_hull_from_stack(chain, cls_name))

    final_objects.sort(key=lambda d: d['x']) 
    return final_objects

# ==========================================
# 3. INFERENCE
# ==========================================
def run_sliding_window_inference(model, color_img):
    gray_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY)
    h_img, w_img = gray_img.shape
    
    TILE_SIZE, OVERLAP = 640, 160 
    STRIDE = TILE_SIZE - OVERLAP
    all_detections = []
    
    print(f"⚡ Scanning {w_img}x{h_img}...")

    for y in tqdm(range(0, h_img, STRIDE)):
        for x in range(0, w_img, STRIDE):
            x_end, y_end = min(w_img, x + TILE_SIZE), min(h_img, y + TILE_SIZE)
            if x_end - x < 50 or y_end - y < 50: continue

            tile_3ch = cv2.cvtColor(gray_img[y:y_end, x:x_end], cv2.COLOR_GRAY2BGR)
            results = model(tile_3ch, conf=0.15, imgsz=640, verbose=False, device=0)
            
            if not results[0].boxes: continue

            for box in results[0].boxes:
                coords = box.xyxy[0].cpu().numpy()
                cls_id = int(box.cls[0].item())
                label = CLASS_MAP.get(cls_id, 'Unknown')
                global_box = np.array([coords[0] + x, coords[1] + y, coords[2] + x, coords[3] + y])
                all_detections.append({'label': label, 'box': global_box})
                
    return all_detections

# ==========================================
# 4. FINAL PIPELINE (Pythagorean No-Overlap)
# ==========================================
def run_metrology():
    print("🚀 RUNNING FINAL METROLOGY (No-Overlap Logic)...")
    
    if not os.path.exists(MODEL_PATH): print("❌ Model not found"); return
    img_color = cv2.imread(IMAGE_PATH) 
    if img_color is None: print("❌ Image not found"); return
    
    model = YOLO(MODEL_PATH)
    raw_fragments = run_sliding_window_inference(model, img_color)
    if not raw_fragments: return

    structures = smart_vertical_stitch(raw_fragments)
    all_tops_y = []

    # Draw Contours
    for s in structures:
        all_tops_y.append(s['top'][1])
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        cv2.polylines(img_color, [s['contour']], True, color, 2)
        cv2.circle(img_color, s['top'], DOT_RADIUS, (0, 0, 255), -1)

    # Measurement Loop with Pythagorean BC Logic
    measurement_idx = 0
    font, f_scale, f_thick, pad = cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2, 4

    for i in range(len(structures) - 1):
        cur, nxt = structures[i], structures[i+1]
        
        if cur['label'] == 'Cathode' and nxt['label'] == 'Anode':
            x1, y1 = cur['top']  # Cathode Tip
            x2, y2 = nxt['top']  # Anode Tip
            
            # Draw ONLY Vertical Component (BC) to reduce clutter
            cv2.line(img_color, (x2, y1), (x2, y2), (255, 0, 255), 2)
            
            dist_bc = abs(y1 - y2) * MICRONS_PER_PIXEL
            text = f"BC: {dist_bc:.1f}"
            
            measurement_idx += 1
            (tw, th), base = cv2.getTextSize(text, font, f_scale, f_thick)
            
            # INCREASED OFFSET to prevent overlap
            t_x = x2 + 12
            t_y = ((y1 + y2) // 2) + (45 if measurement_idx % 2 == 0 else -45)

            cv2.rectangle(img_color, (t_x-pad, t_y-th-pad), (t_x+tw+pad, t_y+base+pad), (0,0,0), -1)
            cv2.putText(img_color, text, (t_x, t_y), font, f_scale, (255,255,255), f_thick)

    # Top Reference Line
    if all_tops_y:
        y_ref = min(all_tops_y) 
        cv2.line(img_color, (0, y_ref), (img_color.shape[1], y_ref), (0, 255, 0), 2)

    cv2.imwrite(OUTPUT_PATH, img_color)
    plt.figure(figsize=(20, 15))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

run_metrology()

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\a.tif"
# Ensure this points to your segmentation model (e.g., best.pt from a segment task)
MODEL_PATH = r"E:\Prithu\final new output\runs\segment\train\weights\best.pt" 
OUTPUT_PATH = r"E:\Prithu\final new output\a1.png"

# ✅ SCALING & VISUALS
MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 4

# ==========================================
# 2. GEOMETRY & SEGMENTATION STITCHING
# ==========================================
def get_precise_tip(hull, x_min, y_min, x_max, y_max):
    points = hull[:, 0, :]
    h = y_max - y_min
    
    # Isolate top 15% for tip detection
    roi_height = min(60, h * 0.15) 
    cutoff_y = y_min + roi_height
    top_zone_points = points[points[:, 1] <= cutoff_y]
    
    if len(top_zone_points) == 0:
        top_zone_points = points

    min_y = np.min(top_zone_points[:, 1])
    peak_region = top_zone_points[top_zone_points[:, 1] <= (min_y + 3)]
    
    if len(peak_region) > 0:
        avg_x = int(np.mean(peak_region[:, 0]))
        avg_y = int(np.mean(peak_region[:, 1]))
        return (avg_x, avg_y)
    
    return tuple(points[np.argmin(points[:, 1])])

def create_hull_from_segments(stack, label):
    """Creates a unified polygon from all segment points in a chain."""
    all_points = []
    for item in stack:
        # item['points'] contains the global polygon coordinates from the mask
        all_points.extend(item['points'])
    
    points_array = np.array(all_points, dtype=np.int32)
    hull = cv2.convexHull(points_array)
    
    # Calculate bounding info for tip detection logic
    x, y, w, h = cv2.boundingRect(hull)
    top_point = get_precise_tip(hull, x, y, x+w, y+h)
        
    return {
        'label': label,
        'contour': hull,
        'box': [x, y, x+w, y+h], # Used for gap logic in next iteration
        'top': top_point,
        'x': top_point[0]
    }

def smart_vertical_stitch(raw_detections):
    final_objects = []

    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        # Sort by the top Y of the segment points
        fragments.sort(key=lambda d: np.min(d['points'][:, 1])) 
        active_chains = [] 

        for frag in fragments:
            best_chain_idx = -1
            best_score = -float('inf') 
            f_pts = frag['points']
            f_y_min, f_y_max = np.min(f_pts[:, 1]), np.max(f_pts[:, 1])
            f_x_min, f_x_max = np.min(f_pts[:, 0]), np.max(f_pts[:, 0])
            
            for i, chain in enumerate(active_chains):
                last_piece = chain[-1]
                l_pts = last_piece['points']
                l_y_max = np.max(l_pts[:, 1])
                l_x_min, l_x_max = np.min(l_pts[:, 0]), np.max(l_pts[:, 0])
                
                gap = f_y_min - l_y_max
                max_allowed_gap = 400 if cls_name == 'Cathode' else 250
                
                if 0 <= gap <= max_allowed_gap:
                    # Check horizontal overlap of polygons
                    overlap_start = max(f_x_min, l_x_min)
                    overlap_end = min(f_x_max, l_x_max)
                    overlap_len = max(0, overlap_end - overlap_start)
                    
                    if overlap_len > 0:
                        f_center = (f_x_min + f_x_max) / 2
                        l_center = (l_x_min + l_x_max) / 2
                        center_dist = abs(f_center - l_center)
                        
                        score = overlap_len - (center_dist * 4.0) 
                        if score > best_score:
                            best_score = score
                            best_chain_idx = i

            if best_chain_idx != -1:
                active_chains[best_chain_idx].append(frag)
            else:
                active_chains.append([frag])

        for chain in active_chains:
            total_pts = np.vstack([c['points'] for c in chain])
            y_min, y_max = np.min(total_pts[:, 1]), np.max(total_pts[:, 1])
            if len(chain) == 1 and (y_max - y_min) < 30:
                continue 
            
            final_objects.append(create_hull_from_segments(chain, cls_name))

    final_objects.sort(key=lambda d: d['x']) 
    return final_objects

# ==========================================
# 3. SEGMENTATION INFERENCE
# ==========================================
def run_sliding_window_inference(model, color_img):
    h_img, w_img = color_img.shape[:2]
    TILE_SIZE, OVERLAP = 640, 160 
    STRIDE = TILE_SIZE - OVERLAP
    all_detections = []
    
    print(f"⚡ Scanning {w_img}x{h_img}...")

    for y in tqdm(range(0, h_img, STRIDE)):
        for x in range(0, w_img, STRIDE):
            x_end, y_end = min(w_img, x + TILE_SIZE), min(h_img, y + TILE_SIZE)
            if x_end - x < 50 or y_end - y < 50: continue

            tile = color_img[y:y_end, x:x_end]
            # Use .predict for segmentation tasks
            results = model.predict(tile, conf=0.15, imgsz=640, verbose=False)
            
            # results[0].masks contains the polygons
            if results[0].masks is None: continue

            for i, mask in enumerate(results[0].masks.xy):
                # Convert normalized/local points to global coordinates
                points = mask.copy()
                points[:, 0] += x
                points[:, 1] += y
                
                cls_id = int(results[0].boxes.cls[i].item())
                label = CLASS_MAP.get(cls_id, 'Unknown')
                
                all_detections.append({'label': label, 'points': points})
                
    return all_detections

# ==========================================
# 4. FINAL PIPELINE
# ==========================================
def run_metrology():
    print("🚀 RUNNING SEGMENTATION METROLOGY...")
    
    if not os.path.exists(MODEL_PATH): print(f"❌ Model not found at {MODEL_PATH}"); return
    img_color = cv2.imread(IMAGE_PATH) 
    if img_color is None: print("❌ Image not found"); return
    
    model = YOLO(MODEL_PATH)
    raw_fragments = run_sliding_window_inference(model, img_color)
    if not raw_fragments: return

    structures = smart_vertical_stitch(raw_fragments)
    all_tops_y = []

    # Draw Contours (Polygons)
    for s in structures:
        all_tops_y.append(s['top'][1])
        # Anode = Green, Cathode = Orange
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        
        # ✅ Draw the actual segmented polygon
        cv2.drawContours(img_color, [s['contour']], -1, color, 2)
        cv2.circle(img_color, s['top'], DOT_RADIUS, (0, 0, 255), -1)

    # Measurement Loop
    measurement_idx = 0
    font, f_scale, f_thick, pad = cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2, 4

    for i in range(len(structures) - 1):
        cur, nxt = structures[i], structures[i+1]
        if cur['label'] == 'Cathode' and nxt['label'] == 'Anode':
            x1, y1 = cur['top']
            x2, y2 = nxt['top']
            
            cv2.line(img_color, (x2, y1), (x2, y2), (255, 0, 255), 2)
            dist_bc = abs(y1 - y2) * MICRONS_PER_PIXEL
            text = f"BC: {dist_bc:.1f}"
            
            measurement_idx += 1
            (tw, th), base = cv2.getTextSize(text, font, f_scale, f_thick)
            t_x, t_y = x2 + 12, ((y1 + y2) // 2) + (45 if measurement_idx % 2 == 0 else -45)

            cv2.rectangle(img_color, (t_x-pad, t_y-th-pad), (t_x+tw+pad, t_y+base+pad), (0,0,0), -1)
            cv2.putText(img_color, text, (t_x, t_y), font, f_scale, (255,255,255), f_thick)

    if all_tops_y:
        y_ref = min(all_tops_y) 
        cv2.line(img_color, (0, y_ref), (img_color.shape[1], y_ref), (0, 255, 0), 2)

    cv2.imwrite(OUTPUT_PATH, img_color)
    plt.figure(figsize=(20, 15))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

run_metrology()

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\a.tif"
MODEL_PATH = r"E:\Prithu\final new output\runs\segment\train\weights\best.pt" 
OUTPUT_PATH = r"E:\Prithu\final new output\a1.png"

MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 5

# Class-Specific Thresholds
CONF_THRESH_ANODE = 0.55   
CONF_THRESH_CATHODE = 0.20 

# ==========================================
# 2. REFINED GEOMETRY (Fixed NameError)
# ==========================================
def get_central_peak(pts, hull_box):
    """Finds the highest point within the central 30% of the electrode width."""
    x, y, w, h = hull_box
    # Define width clearly to avoid NameError
    width = w 
    center_x = x + (width / 2)
    
    # Corridor narrowed to 30% (15% each side of center)
    corridor_min = center_x - (width * 0.15)
    corridor_max = center_x + (width * 0.15)
    
    central_pts = pts[(pts[:, 0] >= corridor_min) & (pts[:, 0] <= corridor_max)]
    
    if len(central_pts) == 0:
        return tuple(pts[np.argmin(pts[:, 1])].astype(int))
        
    return tuple(central_pts[np.argmin(central_pts[:, 1])].astype(int))

def create_hull_from_stack(stack, label):
    all_points = np.vstack([item['mask_points'] for item in stack])
    hull = cv2.convexHull(all_points.astype(np.int32))
    
    bx, by, bw, bh = cv2.boundingRect(hull)
    # Pass the bounding box to the peak detector
    top_point = get_central_peak(all_points, (bx, by, bw, bh))
        
    return {
        'label': label,
        'contour': hull,
        'top': top_point,
        'x': top_point[0],
        'y': top_point[1],
        'box': [bx, by, bx+bw, by+bh]
    }

def smart_vertical_stitch(raw_detections):
    final_objects = []
    for cls_name in ['Anode', 'Cathode']:
        fragments = [d for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        fragments.sort(key=lambda d: d['box'][1]) 
        active_chains = [] 

        for frag in fragments:
            assigned = False
            f_y_top = frag['box'][1]
            f_x_mid = (frag['box'][0] + frag['box'][2]) / 2
            
            for chain in active_chains:
                last = chain[-1]
                gap = f_y_top - last['box'][3]
                x_drift = abs(f_x_mid - ((last['box'][0] + last['box'][2]) / 2))
                
                # vertical gap tolerance for 8k images
                if 0 <= gap <= 450 and x_drift < 60:
                    chain.append(frag)
                    assigned = True
                    break
            
            if not assigned:
                active_chains.append([frag])

        for chain in active_chains:
            final_objects.append(create_hull_from_stack(chain, cls_name))

    final_objects.sort(key=lambda d: d['x']) 
    return final_objects

# ==========================================
# 3. INFERENCE
# ==========================================
def run_sliding_window_inference(model, color_img):
    h_img, w_img = color_img.shape[:2]
    TILE_SIZE, OVERLAP = 800, 200 
    STRIDE = TILE_SIZE - OVERLAP
    all_detections = []
    
    for y in tqdm(range(0, h_img, STRIDE)):
        for x in range(0, w_img, STRIDE):
            y_end, x_end = min(h_img, y + TILE_SIZE), min(w_img, x + TILE_SIZE)
            tile = color_img[y:y_end, x:x_end]
            if tile.shape[0] < 50 or tile.shape[1] < 50: continue

            results = model.predict(tile, conf=0.10, imgsz=TILE_SIZE, verbose=False)
            
            if not results[0].masks: continue

            for i, mask in enumerate(results[0].masks.xy):
                cls_id = int(results[0].boxes.cls[i])
                conf = float(results[0].boxes.conf[i])
                
                target_thresh = CONF_THRESH_CATHODE if cls_id == 0 else CONF_THRESH_ANODE
                if conf < target_thresh: continue

                pts = mask.copy()
                pts[:, 0] += x
                pts[:, 1] += y
                
                coords = results[0].boxes.xyxy[i].cpu().numpy()
                global_box = [coords[0]+x, coords[1]+y, coords[2]+x, coords[3]+y]
                
                all_detections.append({
                    'label': CLASS_MAP[cls_id], 
                    'box': global_box, 
                    'mask_points': pts
                })
    return all_detections

# ==========================================
# 4. MAIN
# ==========================================
def run_metrology():
    img_color = cv2.imread(IMAGE_PATH) 
    if img_color is None: return
    
    model = YOLO(MODEL_PATH)
    raw = run_sliding_window_inference(model, img_color)
    structures = smart_vertical_stitch(raw)

    font, f_scale, f_thick = cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2
    
    for i in range(len(structures)):
        s = structures[i]
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        cv2.drawContours(img_color, [s['contour']], -1, color, 2)
        cv2.circle(img_color, s['top'], DOT_RADIUS, (0, 0, 255), -1)

        if i < len(structures) - 1:
            nxt = structures[i+1]
            if s['label'] == 'Cathode' and nxt['label'] == 'Anode':
                y_cath, y_anod = s['top'][1], nxt['top'][1]
                v_x = nxt['top'][0] 
                
                dist_um = abs(y_cath - y_anod) * MICRONS_PER_PIXEL
                cv2.line(img_color, (v_x, y_cath), (v_x, y_anod), (255, 0, 255), 3)
                
                text = f"{dist_um:.1f} um"
                (tw, th), _ = cv2.getTextSize(text, font, f_scale, f_thick)
                tx, ty = v_x + 15, (y_cath + y_anod) // 2
                cv2.rectangle(img_color, (tx-5, ty-th-5), (tx+tw+5, ty+5), (0,0,0), -1)
                cv2.putText(img_color, text, (tx, ty), font, f_scale, (255, 255, 255), f_thick)

    cv2.imwrite(OUTPUT_PATH, img_color)
    plt.figure(figsize=(20, 12))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

if __name__ == "__main__":
    run_metrology()

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\a.tif"
MODEL_PATH = r"E:\Prithu\final new output\runs\segment\train\weights\best.pt" 
OUTPUT_PATH = r"E:\Prithu\final new output\a1.png"

MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 4

# Class-specific thresholds to address missing Cathodes
CONF_CATHODE = 0.20 
CONF_ANODE = 0.50

# ==========================================
# 2. TIP DETECTION LOGIC
# ==========================================
def get_precise_tip(hull, box):
    x, y, w, h = box
    points = hull[:, 0, :]
    
    # Isolate top 15% for tip detection as per your original code
    roi_height = min(60, h * 0.15) 
    cutoff_y = y + roi_height
    top_zone_points = points[points[:, 1] <= cutoff_y]
    
    if len(top_zone_points) == 0:
        top_zone_points = points

    min_y = np.min(top_zone_points[:, 1])
    peak_region = top_zone_points[top_zone_points[:, 1] <= (min_y + 3)]
    
    if len(peak_region) > 0:
        avg_x = int(np.mean(peak_region[:, 0]))
        avg_y = int(np.mean(peak_region[:, 1]))
        return (avg_x, avg_y)
    
    return tuple(points[np.argmin(points[:, 1])])

# ==========================================
# 3. INFERENCE (SEGMENTATION BASED)
# ==========================================
def run_metrology():
    img_color = cv2.imread(IMAGE_PATH)
    if img_color is None: return
    
    model = YOLO(MODEL_PATH)
    h_img, w_img = img_color.shape[:2]
    
    TILE_SIZE, OVERLAP = 640, 160 
    STRIDE = TILE_SIZE - OVERLAP
    
    structures = []
    
    print(f"⚡ Scanning {w_img}x{h_img}...")
    for y in tqdm(range(0, h_img, STRIDE)):
        for x in range(0, w_img, STRIDE):
            x_end, y_end = min(w_img, x + TILE_SIZE), min(h_img, y + TILE_SIZE)
            tile = img_color[y:y_end, x:x_end]
            if tile.shape[0] < 50 or tile.shape[1] < 50: continue

            results = model.predict(tile, conf=min(CONF_CATHODE, CONF_ANODE), imgsz=640, verbose=False)
            
            if results[0].masks is not None:
                for i, mask in enumerate(results[0].masks.xy):
                    cls_id = int(results[0].boxes.cls[i])
                    conf = float(results[0].boxes.conf[i])
                    
                    # Apply class-specific threshold
                    thresh = CONF_CATHODE if cls_id == 0 else CONF_ANODE
                    if conf < thresh: continue
                    
                    # Convert to global coordinates
                    pts = mask.copy()
                    pts[:, 0] += x
                    pts[:, 1] += y
                    
                    # Generate Hull and Tip for this specific fragment
                    hull = cv2.convexHull(pts.astype(np.int32))
                    bx, by, bw, bh = cv2.boundingRect(hull)
                    tip = get_precise_tip(hull, (bx, by, bw, bh))
                    
                    structures.append({
                        'label': CLASS_MAP[cls_id],
                        'contour': hull,
                        'top': tip,
                        'x': tip[0]
                    })

    # Sort fragments left-to-right
    structures.sort(key=lambda s: s['x'])

    # ==========================================
    # 4. DISTANCE CALCULATION (VERTICAL ONLY)
    # ==========================================
    font, f_scale, f_thick, pad = cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2, 4
    
    for i in range(len(structures) - 1):
        cur, nxt = structures[i], structures[i+1]
        
        # Color: Anode = Green, Cathode = Orange/Blue
        color = (0, 255, 0) if cur['label'] == 'Anode' else (0, 140, 255)
        cv2.drawContours(img_color, [cur['contour']], -1, color, 2)
        cv2.circle(img_color, cur['top'], DOT_RADIUS, (0, 0, 255), -1)

        if cur['label'] == 'Cathode' and nxt['label'] == 'Anode':
            x1, y1 = cur['top']
            x2, y2 = nxt['top']
            
            # Vertical measurement line lock
            cv2.line(img_color, (x2, y1), (x2, y2), (255, 0, 255), 2)
            
            dist_bc = abs(y1 - y2) * MICRONS_PER_PIXEL
            text = f"BC: {dist_bc:.1f}"
            
            (tw, th), base = cv2.getTextSize(text, font, f_scale, f_thick)
            t_x, t_y = x2 + 12, (y1 + y2) // 2
            
            cv2.rectangle(img_color, (t_x-pad, t_y-th-pad), (t_x+tw+pad, t_y+base+pad), (0,0,0), -1)
            cv2.putText(img_color, text, (t_x, t_y), font, f_scale, (255,255,255), f_thick)

    # Draw last structure
    if structures:
        last = structures[-1]
        color = (0, 255, 0) if last['label'] == 'Anode' else (0, 140, 255)
        cv2.drawContours(img_color, [last['contour']], -1, color, 2)
        cv2.circle(img_color, last['top'], DOT_RADIUS, (0, 0, 255), -1)

    cv2.imwrite(OUTPUT_PATH, img_color)
    plt.figure(figsize=(20, 15))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

run_metrology()

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\250115_HS24020-01_R2_Kachel_200x_Messungen.tif"
MODEL_PATH = r"c:\Users\06877\AppData\Local\Programs\Microsoft VS Code\best_vertical_detect.pt" 
OUTPUT_PATH = r"E:\Prithu\final new output\Anode_Break_Cathode_Merge_Final_anode box break.png"

MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 4

# ==========================================
# 2. GEOMETRY & PRECISION HELPERS
# ==========================================
def get_x_overlap_and_center_dist(box_top, box_bottom):
    tx1, ty1, tx2, ty2 = box_top
    bx1, by1, bx2, by2 = box_bottom
    overlap_len = max(0, min(tx2, bx2) - max(tx1, bx1))
    center_dist = abs(((tx1 + tx2) / 2) - ((bx1 + bx2) / 2))
    return overlap_len, center_dist

def get_precise_tip(hull, box):
    x, y, w, h = box
    points = hull[:, 0, :]
    roi_height = min(60, h * 0.15) 
    cutoff_y = y + roi_height
    top_zone_points = points[points[:, 1] <= cutoff_y]
    if len(top_zone_points) == 0: top_zone_points = points
    min_y = np.min(top_zone_points[:, 1])
    peak_region = top_zone_points[top_zone_points[:, 1] <= (min_y + 3)]
    if len(peak_region) > 0:
        return (int(np.mean(peak_region[:, 0])), int(np.mean(peak_region[:, 1])))
    return tuple(points[np.argmin(points[:, 1])])

def create_hull_from_stack(stack, label):
    all_points = []
    for item in stack:
        x1, y1, x2, y2 = item['box']
        all_points.extend([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
    hull = cv2.convexHull(np.array(all_points, dtype=np.int32))
    x, y, w, h = cv2.boundingRect(hull)
    full_box = np.array([x, y, x+w, y+h])
    return {
        'label': label, 'contour': hull, 'box': full_box,
        'top': get_precise_tip(hull, full_box), 'x': full_box[0]
    }

# ==========================================
# 3. ADVANCED STITCHING LOGIC
# ==========================================
def smart_vertical_stitch(raw_detections):
    """
    LOGIC:
    - Special merge for broken Cathodes (increased gap tolerance).
    - Special breakage for oversized Anodes (prevents horizontal overlap).
    """
    final_objects = []

    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        # Calculate average box height for Anode breakage logic
        avg_h = np.mean([f['box'][3] - f['box'][1] for f in fragments])
        height_limit = avg_h * 1.5 

        fragments.sort(key=lambda d: d['box'][1]) 
        active_chains = [] 

        for frag in fragments:
            f_h = frag['box'][3] - frag['box'][1]
            best_chain_idx = -1
            best_score = -float('inf') 
            
            # Forced breakage logic: If Anode is huge, start a new chain below
            is_oversized = (cls_name == 'Anode') and (f_h > height_limit)

            if not is_oversized:
                for i, chain in enumerate(active_chains):
                    last_piece = chain[-1]
                    gap = frag['box'][1] - last_piece['box'][3]
                    
                    # Cathodes get a wider gap search (400px), Anodes are tighter (250px)
                    max_gap = 400 if cls_name == 'Cathode' else 250
                    if gap > max_gap: continue 
                    
                    overlap, center_dist = get_x_overlap_and_center_dist(frag['box'], last_piece['box'])
                    
                    if overlap > 0:
                        # Stricter penalty (4.0) to prevent column jumping
                        score = overlap - (center_dist * 4.0) 
                        if score > best_score:
                            best_score = score
                            best_chain_idx = i

            if best_chain_idx != -1:
                active_chains[best_chain_idx].append(frag)
            else:
                active_chains.append([frag])

        for chain in active_chains:
            if len(chain) == 1 and (chain[0]['box'][3] - chain[0]['box'][1]) < 30: continue 
            final_objects.append(create_hull_from_stack(chain, cls_name))

    final_objects.sort(key=lambda d: d['x']) 
    return final_objects

# ==========================================
# 4. RUN PIPELINE
# ==========================================
def run_metrology():
    img_color = cv2.imread(IMAGE_PATH) 
    model = YOLO(MODEL_PATH)
    
    # Inference using Sliding Window to catch small broken pieces
    h_img, w_img = img_color.shape[:2]
    raw_frags = []
    for y in tqdm(range(0, h_img, 480)):
        for x in range(0, w_img, 480):
            tile = img_color[y:min(h_img, y+640), x:min(w_img, x+640)]
            if tile.shape[0] < 50: continue
            results = model(tile, conf=0.10, imgsz=640, verbose=False) # Low conf for breakage
            for box in results[0].boxes:
                c = box.xyxy[0].cpu().numpy()
                raw_frags.append({'label': CLASS_MAP[int(box.cls[0])], 
                                  'box': np.array([c[0]+x, c[1]+y, c[2]+x, c[3]+y])})

    structures = smart_vertical_stitch(raw_frags)
    measurement_idx = 0

    for i, s in enumerate(structures):
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        cv2.polylines(img_color, [s['contour']], True, color, 2)
        cv2.circle(img_color, s['top'], DOT_RADIUS, (0, 0, 255), -1)

        # MEASURE: Cathode (Left) -> Anode (Right) - Pythagorean BC
        if i < len(structures) - 1:
            nxt = structures[i+1]
            if s['label'] == 'Cathode' and nxt['label'] == 'Anode':
                x1, y1 = s['top']; x2, y2 = nxt['top']
                cv2.line(img_color, (x2, y1), (x2, y2), (255, 0, 255), 2)
                dist = abs(y1 - y2) * MICRONS_PER_PIXEL
                measurement_idx += 1
                offset = 45 if measurement_idx % 2 == 0 else -45
                cv2.putText(img_color, f"BC:{dist:.1f}", (x2+12, ((y1+y2)//2)+offset), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    cv2.imwrite(OUTPUT_PATH, img_color)
    plt.figure(figsize=(20, 10)); plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)); plt.show()

run_metrology()

In [ ]:
import os
import shutil
from ultralytics import YOLO

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_YAML = r"E:\Prithu\Final Yolo_TILED\data.yaml"

PROJECT_DIR = r"E:\Prithu\Yolo\Final_Training_Run"
RUN_NAME = "Microscopy_Polygon_Seg_DualGPU"

# ==========================================
# 2. MAIN EXECUTION
# ==========================================
if __name__ == '__main__':
    # --------------------------------------
    # STEP 1: LOAD MODEL
    # --------------------------------------
    print("🧠 Loading YOLOv8-Medium SEGMENTATION model...")
    model = YOLO("yolov8m-seg.pt") 

    # --------------------------------------
    # STEP 2: TRAIN (Dual GPU)
    # --------------------------------------
    print(f"🚀 Starting Multi-GPU Polygon Segmentation Training...")
    
    model.train(
        data=DATA_YAML,
        task='segment',

        # 🟢 MULTI-GPU SETTING
        device=1,      # Uses both GPU 0 and GPU 1
        
        # 🟢 Increased Batch Size (2x GPUs = More VRAM)
        batch=32,           # Increased from 16 to 32 for speed

        epochs=250,
        imgsz=640,
        workers=16,         # Increased workers to feed both GPUs
        cache=True,
        
        # --- OPTIMIZATION ---
        optimizer="AdamW",
        lr0=0.001,
        patience=50,
        cos_lr=True,
        
        # --- AUGMENTATIONS (Slant Friendly) ---
        degrees=2.0,        
        shear=2.0,          
        perspective=0.0001, 
        fliplr=0.5,
        flipud=0.5,
        scale=0.5,          
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, # Gray safety
        
        # ✅ STRUCTURE
        mosaic=1.0,         
        mixup=0.1,
        copy_paste=0.1,    
        
        # 🟢 SEGMENTATION SPECIFIC
        overlap_mask=True,
        mask_ratio=4,       
        
        # --- OUTPUT ---
        project=PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True
    )

    # --------------------------------------
    # STEP 3: VALIDATION
    # --------------------------------------
    # Validation usually runs on a single device even if training was multi-gpu
    print("\n📊 Running Validation...")
    
    metrics = model.val(
        data=DATA_YAML,
        split='val',        
        imgsz=640,
        batch=32,
        device=1, # Val on main GPU is fine
        
        save=True,          
        plots=True,         
        save_conf=True,     
        conf=0.001,         
        iou=0.6,            
        max_det=300,        
        
        project=PROJECT_DIR,
        name=f"{RUN_NAME}_VALIDATION"
    )

    print(f"Mask mAP@50:    {metrics.seg.map50:.4f}")
    
    # Save to current folder
    best_weight_source = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")
    local_save_path = os.path.join(os.getcwd(), "best_microscopy_seg_dual.pt")
    
    if os.path.exists(best_weight_source):
        shutil.copy(best_weight_source, local_save_path)
        print(f"\n✅ Best model saved locally as: {local_save_path}")

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\a.tif"
MODEL_PATH = r"E:\Prithu\final new output\runs\segment\train\weights\best.pt" 
MODEL_VERT_PATH = r"E:\Prithu\best_workstation_XL.pt"
OUTPUT_PATH = r"E:\Prithu\a1.png"

MICRONS_PER_PIXEL = 1.88424 
TILT_THRESHOLD = 3.0 
DOT_RADIUS = 5

# ==========================================
# 2. REFINED GEOMETRY HELPERS
# ==========================================
def get_precise_tip(hull):
    """Finds the absolute highest central point, even if tilted."""
    pts = hull.reshape(-1, 2)
    min_y = np.min(pts[:, 1])
    
    # Take all points within the top 5 pixels of the peak
    top_region = pts[pts[:, 1] <= (min_y + 5)]
    
    # Use the average (centroid) of the top region to center the dot
    if len(top_region) > 0:
        avg_x = int(np.mean(top_region[:, 0]))
        avg_y = int(np.mean(top_region[:, 1]))
        return (avg_x, avg_y)
    return tuple(pts[np.argmin(pts[:, 1])])

def get_tilt_angle(contour):
    pts = contour.reshape(-1, 2)
    if len(pts) < 5: return 0.0
    [vx, vy, x, y] = cv2.fitLine(pts, cv2.DIST_L2, 0, 0.01, 0.01)
    angle_deg = math.degrees(math.atan2(vy, vx))
    deviation = 90 - abs(angle_deg) if angle_deg > 0 else 90 - abs(abs(angle_deg))
    return abs(deviation)

# ==========================================
# 3. GLOBAL HISTOGRAM MERGE LOGIC
# ==========================================
def global_column_merge(raw_detections, x_threshold=45):
    """Groups broken fragments into columns based on horizontal alignment."""
    final_objects = []
    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        for f in fragments:
            f['cx'] = (f['box'][0] + f['box'][2]) / 2

        fragments.sort(key=lambda x: x['cx'])
        columns = []
        if fragments:
            current_col = [fragments[0]]
            for i in range(1, len(fragments)):
                if abs(fragments[i]['cx'] - current_col[-1]['cx']) < x_threshold:
                    current_col.append(fragments[i])
                else:
                    columns.append(current_col)
                    current_col = [fragments[i]]
            columns.append(current_col)

        for col in columns:
            all_pts = []
            for f in col:
                x1, y1, x2, y2 = f['box']
                all_pts.extend([[x1,y1], [x2,y1], [x2,y2], [x1,y2]])
            
            hull = cv2.convexHull(np.array(all_pts, dtype=np.int32))
            top_pt = get_precise_tip(hull)
            
            final_objects.append({
                'label': cls_name, 'contour': hull, 'top': top_pt, 'x': top_pt[0]
            })

    final_objects.sort(key=lambda d: d['x'])
    return final_objects

# ==========================================
# 4. MAIN HYBRID PIPELINE
# ==========================================
def run_hybrid_metrology():
    img = cv2.imread(IMAGE_PATH)
    if img is None: return
    h, w, _ = img.shape
    roi_y = int(h * 0.40)
    roi_img = img[roi_y:h, 0:w]

    # --- SCOUTING ---
    model_seg = YOLO(MODEL_SEG_PATH)
    results_scout = model_seg.predict(roi_img, imgsz=1280, conf=0.25, verbose=False)
    
    angles = [get_tilt_angle(np.array(mask, dtype=np.int32)) for mask in results_scout[0].masks.xy] if results_scout[0].masks else []
    avg_tilt = np.mean(angles) if angles else 0.0
    
    # --- DECISION ---
    if avg_tilt < TILT_THRESHOLD and os.path.exists(MODEL_VERT_PATH):
        model = YOLO(MODEL_VERT_PATH); model_name = "Vertical (Box)"
    else:
        model = model_seg; model_name = "Polygon (Seg)"

    # --- INFERENCE ---
    print(f"🚀 Using {model_name} | Avg Tilt: {avg_tilt:.2f}°")
    results = model.predict(roi_img, imgsz=3200, conf=0.15, verbose=False)
    
    raw_frags = []
    for i, box in enumerate(results[0].boxes):
        c = box.xyxy[0].cpu().numpy()
        raw_frags.append({
            'label': CLASS_MAP.get(int(box.cls[0]), 'Unknown'),
            'box': np.array([c[0], c[1]+roi_y, c[2], c[3]+roi_y])
        })

    # --- MERGE & MEASURE ---
    structures = global_column_merge(raw_frags)
    measurement_idx = 0

    for i, s in enumerate(structures):
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        cv2.polylines(img, [s['contour']], True, color, 3)
        cv2.circle(img, s['top'], DOT_RADIUS, (0, 0, 255), -1)

        if i < len(structures) - 1:
            nxt = structures[i+1]
            if s['label'] == 'Cathode' and nxt['label'] == 'Anode':
                p1, p2 = s['top'], nxt['top']
                
                # Pythagorean BC Logic (Vertical distance only)
                # Drawing line from level of Cathode to Anode tip
                cv2.line(img, (p2[0], p1[1]), (p2[0], p2[1]), (255, 0, 255), 3)
                
                dist_bc = abs(p1[1] - p2[1]) * MICRONS_PER_PIXEL
                measurement_idx += 1
                offset = 75 if measurement_idx % 2 == 0 else -75
                
                text = f"BC:{dist_bc:.1f}"
                (tw, th), b = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
                tx, ty = p2[0] + 15, ((p1[1] + p2[1]) // 2) + offset
                
                cv2.rectangle(img, (tx-5, ty-th-5), (tx+tw+5, ty+b+5), (0,0,0), -1)
                cv2.putText(img, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv2.imwrite(OUTPUT_PATH, img)
    print(f"💾 Saved to: {OUTPUT_PATH}")
    plt.figure(figsize=(20, 12)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

if __name__ == "__main__":
    run_hybrid_metrology()

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\251202_HS25020-02_R2_Kachel_20x.tif"
MODEL_PATH = r"E:\Prithu\final new output\runs\segment\train\weights\best.pt" 
OUTPUT_PATH = r"E:\Prithu\final new output\Final_Global_Histogram_Merge.png"

MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 6

# ==========================================
# 2. GLOBAL COLUMN MERGING (Captures all missed pieces)
# ==========================================
def global_column_merge(raw_detections, x_threshold=45):
    """
    Groups fragments into columns based on X-centroid similarity.
    Increases the chance of capturing broken tips that were previously missed.
    """
    final_objects = []

    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        # Calculate centroids for all pieces
        for f in fragments:
            f['cx'] = (f['box'][0] + f['box'][2]) / 2

        # Sort strictly Left-to-Right
        fragments.sort(key=lambda x: x['cx'])

        columns = []
        if fragments:
            current_col = [fragments[0]]
            for i in range(1, len(fragments)):
                # If pieces are horizontally aligned, they are the same electrode body
                if abs(fragments[i]['cx'] - current_col[-1]['cx']) < x_threshold:
                    current_col.append(fragments[i])
                else:
                    columns.append(current_col)
                    current_col = [fragments[i]]
            columns.append(current_col)

        # Fuse segments into a single unified electrode
        for col in columns:
            all_pts = []
            for f in col:
                x1, y1, x2, y2 = f['box']
                all_pts.extend([[x1,y1], [x2,y1], [x2,y2], [x1,y2]])
            
            pts_array = np.array(all_pts, dtype=np.int32)
            hull = cv2.convexHull(pts_array)
            
            # Find absolute peak of the entire combined structure
            min_y_idx = np.argmin(hull[:, 0, 1])
            top_pt = tuple(hull[min_y_idx][0])
            
            final_objects.append({
                'label': cls_name,
                'contour': hull,
                'top': top_pt,
                'x': top_pt[0]
            })

    # Final order Left-to-Right for pairing
    final_objects.sort(key=lambda d: d['x'])
    return final_objects

# ==========================================
# 3. HIGH-RES SCANNING & PIPELINE
# ==========================================
def run_metrology():
    img = cv2.imread(IMAGE_PATH)
    if img is None: return
    
    model = YOLO(MODEL_PATH)
    h_img, w_img = img.shape[:2]
    raw_frags = []

    # Scanning with overlap to ensure tips on tile edges aren't missed
    for y in tqdm(range(0, h_img, 480)):
        for x in range(0, w_img, 480):
            tile = img[y:min(h_img, y+640), x:min(w_img, x+640)]
            if tile.shape[0] < 50: continue
            
            # Low confidence to bridge narrow breakage gaps
            results = model.predict(tile, conf=0.10, imgsz=640, verbose=False, device=1)
            
            if results[0].boxes:
                for box in results[0].boxes:
                    c = box.xyxy[0].cpu().numpy()
                    raw_frags.append({
                        'label': CLASS_MAP[int(box.cls[0])],
                        'box': np.array([c[0]+x, c[1]+y, c[2]+x, c[3]+y])
                    })

    # Merge broken pieces globally
    structures = global_column_merge(raw_frags)
    measurement_idx = 0

    for i, s in enumerate(structures):
        # Draw electrode boundary and Tip
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        cv2.polylines(img, [s['contour']], True, color, 2)
        cv2.circle(img, s['top'], DOT_RADIUS, (0, 0, 255), -1)

        # MEASURE: Pairing Cathode to Anode
        if i < len(structures) - 1:
            nxt = structures[i+1]
            if s['label'] == 'Cathode' and nxt['label'] == 'Anode':
                p1, p2 = s['top'], nxt['top']
                
                # Pythagorean BC (Straight Vertical distance from level of A to point C)
                cv2.line(img, (p2[0], p1[1]), (p2[0], p2[1]), (255, 0, 255), THICKNESS)
                
                dist_bc = abs(p1[1] - p2[1]) * MICRONS_PER_PIXEL
                measurement_idx += 1
                
                # LABEL LOGIC: Large alternating offsets for clear visibility
                text = f"{dist_bc:.1f}"
                (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
                
                # Move text further to the side of the line
                text_x = p2[0] + 15 
                # Alternate height to avoid overlap
                text_y = ((p1[1] + p2[1]) // 2) + (80 if measurement_idx % 2 == 0 else -80)

                cv2.rectangle(img, (text_x - 5, text_y - th - 5), (text_x + tw + 5, text_y + baseline + 5), (0,0,0), -1)
                cv2.putText(img, text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv2.imwrite(OUTPUT_PATH, img)
    plt.figure(figsize=(22, 14)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

run_metrology()

In [ ]:
from ultralytics import YOLO

# Path to your completed weights
LAST_WEIGHTS = r"E:\Prithu\Yolo\Final_Training_Run\Microscopy_Polygon_Seg_DualGPU\weights\last.pt"

if __name__ == '__main__':
    print(f"🚀 Starting further training using: {LAST_WEIGHTS}")
    
    # Load the model normally (DO NOT use resume=True in the train call)
    model = YOLO(LAST_WEIGHTS)
    
    # Start a new training session continuing from these weights
    model.train(
        data=r"E:\Prithu\Final Yolo_TILED\data.yaml", # You must specify the data path again
        epochs=250,      # Set how many MORE epochs you want (e.g., another 250)
        patience=50,     # Early stopping if no improvement after 50 epochs
        batch=32,
        device=1,
        imgsz=640
    )

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import math
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\251202_HS25020-02_R2_Kachel_20x.tif"
MODEL_PATH = r"E:\Prithu\final new output\runs\segment\train\weights\best.pt"
OUTPUT_PATH = r"E:\Prithu\final new output\Final_AnodeFix_Metrology.png"

MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
DOT_RADIUS = 5

# ==========================================
# 2. ROBUST COLUMN GROUPING
# ==========================================
def global_column_merge(raw_detections, x_threshold=45):
    final_objects = []

    for cls_name in ['Anode', 'Cathode']:
        fragments = [d.copy() for d in raw_detections if d['label'] == cls_name]
        if not fragments: continue

        for f in fragments:
            f['cx'] = (f['box'][0] + f['box'][2]) / 2

        fragments.sort(key=lambda x: x['cx'])
        columns = []
        if fragments:
            current_col = [fragments[0]]
            for i in range(1, len(fragments)):
                if abs(fragments[i]['cx'] - current_col[-1]['cx']) < x_threshold:
                    current_col.append(fragments[i])
                else:
                    columns.append(current_col)
                    current_col = [fragments[i]]
            columns.append(current_col)

        for col in columns:
            # Sort fragments in this column by Y (top to bottom)
            col.sort(key=lambda f: f['box'][1])
            
            # For Anodes, we strictly want the top-most fragment to define the tip
            # but we use all points for the visual hull
            all_pts = []
            for f in col:
                x1, y1, x2, y2 = f['box']
                all_pts.extend([[x1,y1], [x2,y1], [x2,y2], [x1,y2]])
            
            hull = cv2.convexHull(np.array(all_pts, dtype=np.int32))
            
            # Measurement Point: Absolute highest Y pixel in the column
            min_y_idx = np.argmin(hull[:, 0, 1])
            top_pt = tuple(hull[min_y_idx][0])
            
            final_objects.append({
                'label': cls_name, 'contour': hull, 'top': top_pt, 'x': top_pt[0]
            })

    final_objects.sort(key=lambda d: d['x'])
    return final_objects

# ==========================================
# 3. SCANNING WITH DYNAMIC ANODE LOGIC
# ==========================================
def run_metrology():
    img = cv2.imread(IMAGE_PATH)
    if img is None: return
    model = YOLO(MODEL_PATH)
    h_img, w_img = img.shape[:2]
    raw_frags = []

    print("⚡ Capturing Electrode Geometry...")
    for y in tqdm(range(0, h_img, 480)):
        for x in range(0, w_img, 480):
            tile = img[y:min(h_img, y+640), x:min(w_img, x+640)]
            if tile.shape[0] < 50: continue
            
            # Increased confidence slightly for Anodes to avoid false body segments
            results = model.predict(tile, conf=0.15, imgsz=640, verbose=False, device=0)
            
            if results[0].boxes:
                for box in results[0].boxes:
                    c = box.xyxy[0].cpu().numpy()
                    label = CLASS_MAP.get(int(box.cls[0]), 'Unknown')
                    
                    # 🟢 ANODE REFINEMENT: 
                    # If it's a very long vertical box, it's the body, not the tip.
                    # We keep fragments that look like 'tips' (height < 400px)
                    if label == 'Anode' and (c[3] - c[1]) > 400:
                        continue

                    raw_frags.append({
                        'label': label,
                        'box': np.array([c[0]+x, c[1]+y, c[2]+x, c[3]+y])
                    })

    # Merge fragments into final columns
    structures = global_column_merge(raw_frags)
    measurement_idx = 0

    for i, s in enumerate(structures):
        # Draw clean polygons and peak dots
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        cv2.polylines(img, [s['contour']], True, color, 2)
        cv2.circle(img, s['top'], DOT_RADIUS, (0, 0, 255), -1)

        # MEASURE: Top-to-Top Vertical BC
        if i < len(structures) - 1:
            nxt = structures[i+1]
            if s['label'] == 'Cathode' and nxt['label'] == 'Anode':
                p1_y = s['top'][1]
                p2_x, p2_y = nxt['top']
                
                # Pythagorean BC (Absolute vertical difference)
                dist_bc = abs(p1_y - p2_y) * MICRONS_PER_PIXEL
                measurement_idx += 1
                
                # Label positioning
                text = f"{dist_bc:.1f}"
                (tw, th), b = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
                offset = 85 if measurement_idx % 2 == 0 else -85
                ty = ((p1_y + p2_y) // 2) + offset
                
                cv2.rectangle(img, (p2_x+15, ty-th-5), (p2_x+15+tw, ty+b+5), (0,0,0), -1)
                cv2.putText(img, text, (p2_x+15, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv2.imwrite(OUTPUT_PATH, img)
    plt.figure(figsize=(20, 12)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

run_metrology()

In [ ]:
%matplotlib inline 

import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
IMAGE_PATH = r"E:\Prithu\new images\a.tif"
MODEL_PATH = r"e:\Prithu\final new output\best_vertical_detect.pt" 
OUTPUT_PATH = r"E:\Prithu\final new output\a1_segmented.png"

# ✅ TUNING PARAMETERS
MICRONS_PER_PIXEL = 1.88424 
CLASS_MAP = {0: 'Cathode', 1: 'Anode'}
CONF_CATHODE = 0.18  # Lowered to capture missed cathodes
CONF_ANODE = 0.55    # Strict for anodes
DOT_RADIUS = 6

# ==========================================
# 2. ADVANCED GEOMETRY & CENTRAL PEAK
# ==========================================
def get_central_peak(mask_pts):
    """Finds the highest point (min Y) in the central 30% width of the electrode."""
    x_min, x_max = np.min(mask_pts[:, 0]), np.max(mask_pts[:, 0])
    width = x_max - x_min
    center_x = x_min + (width / 2)
    
    # Corridor narrowed for "central part" measurement
    corridor_min = center_x - (width * 0.15)
    corridor_max = center_x + (width * 0.15)
    
    central_pts = mask_pts[(mask_pts[:, 0] >= corridor_min) & (mask_pts[:, 0] <= corridor_max)]
    
    if len(central_pts) == 0:
        return tuple(mask_pts[np.argmin(mask_pts[:, 1])].astype(int))
        
    return tuple(central_pts[np.argmin(central_pts[:, 1])].astype(int))

def merge_and_stitch(raw_detections, img_shape):
    """Fuses segmentation masks and bridges vertical breakage."""
    h, w = img_shape[:2]
    final_structures = []

    for cls_id, cls_name in CLASS_MAP.items():
        # Create class mask for pixel-perfect merging
        mask_layer = np.zeros((h, w), dtype=np.uint8)
        for det in raw_detections:
            if det['label'] == cls_name:
                cv2.fillPoly(mask_layer, [det['mask'].astype(np.int32)], 255)
        
        # Bridge vertical breakage (Closing operation)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 70))
        mask_layer = cv2.morphologyEx(mask_layer, cv2.MORPH_CLOSE, kernel)
        
        contours, _ = cv2.findContours(mask_layer, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for cnt in contours:
            if cv2.contourArea(cnt) < 300: continue
            
            hull = cv2.convexHull(cnt)
            # Flatten contour for peak detection
            pts = hull.reshape(-1, 2)
            tip = get_central_peak(pts)
            
            final_structures.append({
                'label': cls_name,
                'contour': hull,
                'top': tip,
                'x': tip[0]
            })

    final_structures.sort(key=lambda d: d['x']) 
    return final_structures

# ==========================================
# 3. INFERENCE
# ==========================================
def run_segmentation_inference(model, color_img):
    h_img, w_img = color_img.shape[:2]
    TILE_SIZE, OVERLAP = 640, 180 
    STRIDE = TILE_SIZE - OVERLAP
    all_detections = []
    
    print(f"⚡ Scanning {w_img}x{h_img}...")

    for y in tqdm(range(0, h_img, STRIDE)):
        for x in range(0, w_img, STRIDE):
            y_end, x_end = min(h_img, y + TILE_SIZE), min(w_img, x + TILE_SIZE)
            tile = color_img[y:y_end, x:x_end]
            
            # Predict with lowest common denominator for confidence
            results = model.predict(tile, conf=min(CONF_CATHODE, CONF_ANODE), imgsz=640, verbose=False)
            
            if results[0].masks is not None:
                for i, mask in enumerate(results[0].masks.xy):
                    cls_id = int(results[0].boxes.cls[i])
                    conf = float(results[0].boxes.conf[i])
                    
                    # Class-specific filtering
                    thresh = CONF_CATHODE if cls_id == 0 else CONF_ANODE
                    if conf < thresh: continue
                    
                    global_mask = mask.copy()
                    global_mask[:, 0] += x
                    global_mask[:, 1] += y
                    all_detections.append({'label': CLASS_MAP[cls_id], 'mask': global_mask})
                
    return all_detections

# ==========================================
# 4. MAIN WORKFLOW
# ==========================================
def run_metrology():
    img_color = cv2.imread(IMAGE_PATH) 
    if img_color is None: return
    
    model = YOLO(MODEL_PATH)
    raw_detections = run_segmentation_inference(model, img_color)
    structures = merge_and_stitch(raw_detections, img_color.shape)

    # Styling
    font, f_scale, f_thick, pad = cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2, 5

    for i in range(len(structures)):
        s = structures[i]
        # Anode: Green, Cathode: Orange/Blue
        color = (0, 255, 0) if s['label'] == 'Anode' else (0, 140, 255)
        cv2.drawContours(img_color, [s['contour']], -1, color, 2)
        cv2.circle(img_color, s['top'], DOT_RADIUS, (0, 0, 255), -1)

        # DISTANCE CALCULATION: Cathode Tip to Anode Tip
        if i < len(electrodes := structures) - 1:
            cur, nxt = electrodes[i], electrodes[i+1]
            
            if cur['label'] == 'Cathode' and nxt['label'] == 'Anode':
                y1 = cur['top'][1]  # Cathode Y
                y2 = nxt['top'][1]  # Anode Y
                v_x = nxt['top'][0] # Vertical lock to Anode Center Peak
                
                # Draw Strict Vertical BC Line
                cv2.line(img_color, (v_x, y1), (v_x, y2), (255, 0, 255), 3)
                
                dist_bc = abs(y1 - y2) * MICRONS_PER_PIXEL
                text = f"BC: {dist_bc:.1f} um"
                
                # Visible Label with Background
                (tw, th), base = cv2.getTextSize(text, font, f_scale, f_thick)
                t_x, t_y = v_x + 15, (y1 + y2) // 2
                cv2.rectangle(img_color, (t_x-pad, t_y-th-pad), (t_x+tw+pad, t_y+base+pad), (0,0,0), -1)
                cv2.putText(img_color, text, (t_x, t_y), font, f_scale, (255,255,255), f_thick)

    cv2.imwrite(OUTPUT_PATH, img_color)
    plt.figure(figsize=(20, 15))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

run_metrology()

In [ ]:
pip install pandas

In [ ]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import os

# ==========================================
# ⚙️ CONFIGURATION - UPDATE THESE PATHS
# ==========================================
# Path to your BEST trained model
MODEL_PATH = r"e:\Prithu\final new output\best_vertical_detect.pt" 

# Folder containing images to test
IMAGE_PATH = r"E:\Prithu\new images\a.tif"

# Folder to save results (images + csv)
OUTPUT_FOLDER = r"e:\Prithu\Inference_Results_With_Distances"

# 📏 CALIBRATION
# How many microns is 1 pixel? 
# Set to 1.0 if you only want pixel values.
# Example: If a 1000px scale bar = 500um, then factor is 0.5
MICRONS_PER_PIXEL = 1.72  

# 🎚️ INFERENCE SETTINGS
CONF_THRESHOLD = 0.5  # Only keep high confidence detections (since your mAP is high)
IOU_THRESHOLD = 0.4   # NMS threshold to remove overlapping boxes
# ==========================================

def process_images():
    # Load Model
    print(f"🔄 Loading model from: {MODEL_PATH}")
    model = YOLO(MODEL_PATH)
    
    # Create output directory
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    # Get list of images
    image_files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))]
    
    if not image_files:
        print("❌ No images found in input folder!")
        return

    print(f"🚀 Starting inference on {len(image_files)} images...")

    for img_name in image_files:
        img_path = os.path.join(INPUT_FOLDER, img_name)
        
        # 1. Run Prediction
        results = model.predict(
            img_path, 
            conf=CONF_THRESHOLD, 
            iou=IOU_THRESHOLD, 
            verbose=False
        )[0]
        
        # Load image for drawing
        original_img = cv2.imread(img_path)
        draw_img = original_img.copy()
        
        # 2. Extract Data
        # We need: x1, y1, x2, y2, confidence, class_id
        boxes = results.boxes.data.cpu().numpy() # Returns (x1, y1, x2, y2, conf, cls)
        
        if len(boxes) == 0:
            print(f"⚠️ No detections in {img_name}")
            continue

        # 3. Sort Boxes Left-to-Right based on x1 coordinate
        # This is CRITICAL for sequential distance calculation
        sorted_indices = np.argsort(boxes[:, 0])
        boxes = boxes[sorted_indices]
        
        data_list = []
        
        # 4. Loop through sorted boxes to calculate metrics
        for i in range(len(boxes)):
            x1, y1, x2, y2, conf, cls_id = boxes[i]
            class_name = model.names[int(cls_id)]
            
            # --- MEASUREMENTS ---
            # Width & Height
            pixel_width = x2 - x1
            pixel_height = y2 - y1
            
            real_width = pixel_width * MICRONS_PER_PIXEL
            real_height = pixel_height * MICRONS_PER_PIXEL
            
            # Gap to NEXT box (Horizontal Distance)
            # Distance from current box's RIGHT edge (x2) to next box's LEFT edge (next_x1)
            gap_pixel = 0
            gap_real = 0
            
            if i < len(boxes) - 1:
                next_x1 = boxes[i+1][0]
                gap_pixel = next_x1 - x2
                # If gap is negative, boxes overlap horizontally
                gap_pixel = max(0, gap_pixel) 
                gap_real = gap_pixel * MICRONS_PER_PIXEL

            # Store Data
            data_list.append({
                "ID": i + 1,
                "Class": class_name,
                "Confidence": f"{conf:.4f}",
                "Width_px": f"{pixel_width:.1f}",
                "Width_um": f"{real_width:.2f}",
                "Height_px": f"{pixel_height:.1f}",
                "Height_um": f"{real_height:.2f}",
                "Gap_to_Next_px": f"{gap_pixel:.1f}",
                "Gap_to_Next_um": f"{gap_real:.2f}",
                "x1": x1, "y1": y1, "x2": x2, "y2": y2
            })
            
            # --- VISUALIZATION ---
            # Color: Anode (Red), Cathode (Green) - adjust as needed
            color = (0, 255, 0) if "Cathode" in class_name else (0, 0, 255)
            
            # Draw Bounding Box
            cv2.rectangle(draw_img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            
            # Draw Labels (ID and Width)
            label = f"#{i+1} {class_name} W:{real_width:.1f}um"
            cv2.putText(draw_img, label, (int(x1), int(y1) - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Draw Gap Line (to next box)
            if i < len(boxes) - 1 and gap_pixel > 0:
                next_x1 = boxes[i+1][0]
                mid_y = int((y1 + y2) / 2)
                # Yellow line for gap
                cv2.line(draw_img, (int(x2), mid_y), (int(next_x1), mid_y), (0, 255, 255), 2)
                # Gap Text
                cv2.putText(draw_img, f"{gap_real:.1f}um", (int(x2) + 5, mid_y - 5), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

        # 5. Save Results
        # Save CSV
        df = pd.DataFrame(data_list)
        csv_name = os.path.splitext(img_name)[0] + "_measurements.csv"
        df.to_csv(os.path.join(OUTPUT_FOLDER, csv_name), index=False)
        
        # Save Image
        save_img_name = os.path.splitext(img_name)[0] + "_analyzed.jpg"
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, save_img_name), draw_img)
        
        print(f"✅ Processed {img_name} -> Found {len(boxes)} components")

    print(f"\n🎉 Done! Results saved to: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    process_images()

In [ ]:
%matplotlib inline

import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm

# =========================================================
# 1. CONFIGURATION
# =========================================================

IMAGE_PATH  = r"E:\Prithu\new images\a.tif"
MODEL_PATH  = r"E:\Prithu\final new output\best_vertical_detect.pt"
OUTPUT_PATH = r"E:\Prithu\final new output\a1.png"

MICRONS_PER_PIXEL = 1.88424
CLASS_MAP = {0: "Cathode", 1: "Anode"}

TILE_SIZE = 640
OVERLAP = 160
STRIDE = TILE_SIZE - OVERLAP

RAW_NMS_IOU = 0.6
COLUMN_X_THRESHOLD = 28
MIN_COLUMN_HEIGHT = 40
DOT_RADIUS = 4

# =========================================================
# 2. BASIC GEOMETRY
# =========================================================

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    iw = max(0.0, inter_x2 - inter_x1)
    ih = max(0.0, inter_y2 - inter_y1)

    inter = iw * ih
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)

    return inter / (area_a + area_b - inter + 1e-9)


def box_center_x(box):
    return 0.5 * (box[0] + box[2])


def box_height(box):
    return box[3] - box[1]


def union_box(boxes):
    x1 = min(b[0] for b in boxes)
    y1 = min(b[1] for b in boxes)
    x2 = max(b[2] for b in boxes)
    y2 = max(b[3] for b in boxes)
    return np.array([x1, y1, x2, y2], dtype=np.float32)


# =========================================================
# 3. SLIDING WINDOW INFERENCE
# =========================================================

def run_inference(model, image):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    detections = []

    print(f"Scanning {w}x{h} image...")

    for y in tqdm(range(0, h, STRIDE)):
        for x in range(0, w, STRIDE):

            x_end = min(w, x + TILE_SIZE)
            y_end = min(h, y + TILE_SIZE)

            if x_end - x < 50 or y_end - y < 50:
                continue

            tile = gray[y:y_end, x:x_end]
            tile = cv2.cvtColor(tile, cv2.COLOR_GRAY2BGR)

            results = model(tile, conf=0.15, imgsz=640, verbose=False)

            if not results or not results[0].boxes:
                continue

            for box in results[0].boxes:

                coords = box.xyxy[0].cpu().numpy()
                cls_id = int(box.cls[0].item())
                label = CLASS_MAP.get(cls_id, "Unknown")

                global_box = np.array([
                    coords[0] + x,
                    coords[1] + y,
                    coords[2] + x,
                    coords[3] + y
                ], dtype=np.float32)

                detections.append({
                    "label": label,
                    "box": global_box
                })

    return detections


# =========================================================
# 4. GLOBAL NMS (REMOVE TILE DUPLICATES)
# =========================================================

def global_nms(detections):

    final = []

    for cls in ["Cathode", "Anode"]:

        cls_boxes = [d for d in detections if d["label"] == cls]
        cls_boxes.sort(key=lambda d: box_height(d["box"]), reverse=True)

        keep = []

        for d in cls_boxes:
            if all(iou_xyxy(d["box"], k["box"]) < RAW_NMS_IOU for k in keep):
                keep.append(d)

        final.extend(keep)

    return final


# =========================================================
# 5. COLUMN GROUPING (STABLE VERTICAL STACKS)
# =========================================================

def cluster_columns(detections, label):

    objs = [d for d in detections if d["label"] == label]
    if not objs:
        return []

    objs.sort(key=lambda d: box_center_x(d["box"]))

    columns = []

    for obj in objs:

        cx = box_center_x(obj["box"])
        y1, y2 = obj["box"][1], obj["box"][3]
        placed = False

        for col in columns:

            if abs(cx - col["mean_x"]) < COLUMN_X_THRESHOLD:

                col["boxes"].append(obj)
                col["mean_x"] = np.mean([box_center_x(o["box"]) for o in col["boxes"]])
                col["y_range"] = (
                    min(col["y_range"][0], y1),
                    max(col["y_range"][1], y2)
                )
                placed = True
                break

        if not placed:
            columns.append({
                "boxes": [obj],
                "mean_x": cx,
                "y_range": (y1, y2)
            })

    final_columns = []

    for col in columns:

        col_box = union_box([o["box"] for o in col["boxes"]])

        if box_height(col_box) < MIN_COLUMN_HEIGHT:
            continue

        top_point = (int((col_box[0]+col_box[2])/2), int(col_box[1]))

        final_columns.append({
            "label": label,
            "box": col_box,
            "top": top_point,
            "x": box_center_x(col_box)
        })

    return final_columns


# =========================================================
# 6. STABLE PAIRING (BY COLUMN ORDER)
# =========================================================

def pair_by_order(cathodes, anodes):

    cathodes.sort(key=lambda c: c["x"])
    anodes.sort(key=lambda a: a["x"])

    min_len = min(len(cathodes), len(anodes))

    return [(cathodes[i], anodes[i]) for i in range(min_len)]


# =========================================================
# 7. MAIN METROLOGY PIPELINE
# =========================================================

def run_metrology():

    if not os.path.exists(MODEL_PATH):
        print("Model not found.")
        return

    image = cv2.imread(IMAGE_PATH)
    if image is None:
        print("Image not found.")
        return

    model = YOLO(MODEL_PATH)

    detections = run_inference(model, image)

    if not detections:
        print("No detections found.")
        return

    detections = global_nms(detections)

    cathodes = cluster_columns(detections, "Cathode")
    anodes   = cluster_columns(detections, "Anode")

    pairs = pair_by_order(cathodes, anodes)

    # -----------------------------------------------------
    # DRAWING
    # -----------------------------------------------------

    for obj in cathodes:
        cv2.rectangle(image,
                      (int(obj["box"][0]), int(obj["box"][1])),
                      (int(obj["box"][2]), int(obj["box"][3])),
                      (0,140,255), 2)
        cv2.circle(image, obj["top"], DOT_RADIUS, (0,0,255), -1)

    for obj in anodes:
        cv2.rectangle(image,
                      (int(obj["box"][0]), int(obj["box"][1])),
                      (int(obj["box"][2]), int(obj["box"][3])),
                      (0,255,0), 2)
        cv2.circle(image, obj["top"], DOT_RADIUS, (0,0,255), -1)

    for c, a in pairs:

        x1, y1 = c["top"]
        x2, y2 = a["top"]

        cv2.line(image, (x2, y1), (x2, y2), (255,0,255), 2)

        distance_um = abs(y1 - y2) * MICRONS_PER_PIXEL
        text = f"{distance_um:.1f} um"

        cv2.putText(image,
                    text,
                    (x2 + 10, (y1 + y2)//2),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255,255,255),
                    2)

    cv2.imwrite(OUTPUT_PATH, image)

    plt.figure(figsize=(20,15))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


# =========================================================
# RUN
# =========================================================

run_metrology()